# Bengali EVC v2 — Emotion-Injection Fix

**v1 problem:** the generator scored ~100% on the frozen-SER confusion matrix while
injecting **no audible emotion** — it learned to fool the frozen classifier instead of
actually converting emotion. **v2 fixes this with four changes:**

1. **Online SER** — a second classifier that keeps adapting to the generator's fakes, so
   the generator can't exploit a fixed model. Half the SER loss comes from it.
2. **Prosody-statistics loss** — matches generated energy mean/std/dynamics to the target.
   Signal-level, so a spectral fingerprint can't fake it.
3. **Gradient-Reversal disentanglement** — pushes the content encoder to be emotion-agnostic,
   cutting source-emotion leakage.
4. **Rebalanced losses** — L1 lowered (10→4, 8→3), SER raised (1→3, 1.5→5), cycle lowered
   (5→2), adversarial raised. Emotion pressure now rivals reconstruction.

Toggles: `CFG["use_emotion_fix"]`, `use_online_ser`, `use_grl`. **v1 checkpoints still load**
(new modules warm-start from the reference SER). **Cell 18 reports the HONEST metrics** —
ignore the frozen-SER confusion matrix as a success signal.

---

# Bengali EVC — Training · Inference · Evaluation · Visualization (Kaggle T4×2)

This notebook is a Kaggle adaptation of your dynamic-length, 3-phase EVC system.

## What's new for Kaggle
- **Auto-resume from `/kaggle/input/datasets/yousufasgormumin57/checkpoint-a-i-r/`** every time before training
- **Dataset auto-discovery** anywhere under `/kaggle/input/`
- **Inference-only fast path** — load checkpoint and skip straight to visualisation + audio playback
- No more Colab `files.upload()` — fully Kaggle-native paths

## Workflow
| Goal | Run cells |
|------|-----------|
| Fresh training run from scratch | 1 → 18 (skip cell 14 "inference-only fast-resume") |
| Resume training from input checkpoint | 1 → 13, 15 → 18 (checkpoint auto-loaded inside cell 15) |
| **Inference only** (no training) | 1 → 12, **14**, then 16 → 18 |

## Pipeline
**Phase 1** (1–50): pure reconstruction (mel L1 + energy) — no GAN, no emotion pressure  
**Phase 2** (51–180): controlled neutral→emotion conversion (cycle + content + SER + adv)  
**Phase 3** (181–250): mild sharpening (slightly higher adv + SER weight)

## 1 · Install, imports, reproducibility, device check

Found 1 checkpoint file(s):

── evc_epoch_250.pt  (30.4 MB)
   type: dict, 15 top-level keys
   keys: ['epoch', 'G', 'D', 'SER', 'EMO_FROM_CONTENT', 'SER_ONLINE', 'opt_G_state', 'opt_D_state', 'opt_grl_state', 'opt_ser_online_state', 'history', 'emotion_to_id', 'speaker_to_id', 'stats', 'cfg']
   epoch = 250



In [ ]:
# ============================================================
# BLOCK 1 — Install, imports, reproducibility, device check
# ============================================================

# Kaggle's base image already has these; install only what's missing/silent.
import subprocess, sys
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
_pip("librosa", "soundfile")

import os, json, math, random, shutil, zipfile, copy, time, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa
import librosa.display
import librosa.sequence
import soundfile as sf

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from scipy import signal

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from IPython.display import Audio, display

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    print(f"GPU count: {n_gpu}")
    for i in range(n_gpu):
        prop = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {prop.name}  | {prop.total_memory / 1024**3:.1f} GB")

## 2 · Configuration — Kaggle paths + checkpoint input dataset

Key change vs. your Colab version:
- `OUT_DIR` is under `/kaggle/working/` (writable, persists across the session)
- `CHECKPOINT_INPUT_DIR` points to your uploaded checkpoint dataset

In [ ]:
# ============================================================
# BLOCK 2 — User settings / paths / master CFG  (Kaggle)
# ============================================================

# Where to look for the PROCESSED feature dataset (output of your previous notebook).
# We auto-scan all of /kaggle/input/ to find metadata.csv or the dataset ZIP.
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

# Optional: explicit ZIP path if you uploaded the ZIP exactly as a dataset.
# Set to None to enable auto-discovery instead.
EXPLICIT_ZIP_PATH = None  # e.g. Path("/kaggle/input/my-features/subesco_processed_dataset.zip")

# Checkpoint dataset uploaded by you (Kaggle dataset slug).
# This is read-only; new checkpoints will be written to /kaggle/working/.
CHECKPOINT_INPUT_DIR = Path("/kaggle/input/datasets/yousufasgormumin57/checkpoint-a-i-r")

# Working / output directories (Kaggle writable).
WORK_ROOT  = Path("/kaggle/working")
EXTRACT_DIR = WORK_ROOT / "subesco_features_extracted"
OUT_DIR    = WORK_ROOT / "evc_clean_250_output"
CKPT_DIR   = OUT_DIR / "checkpoints"
PLOT_DIR   = OUT_DIR / "plots"
AUDIO_DIR  = OUT_DIR / "audio"
CACHE_DIR  = OUT_DIR / "dtw_cache"

for p in [EXTRACT_DIR, OUT_DIR, CKPT_DIR, PLOT_DIR, AUDIO_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CFG = {
    # Audio / feature settings — must match extraction notebook
    "sample_rate": 16000,
    "n_fft": 2048,
    "hop_length": 512,
    "win_length": 2048,
    "n_mels": 128,
    "fmin": 0.0,
    "fmax": 3500.0,
    "top_db": 25.0,

    # Silence / blank-frame handling
    # Extra trim layer runs on top of the pre-trim done during extraction —
    # voiced-flag-aware, with edge zeroing for residual silence.
    "trim_silence": True,
    "trim_use_voiced": True,    # use voiced flag (UNION with energy) to find boundaries
    "trim_top_db_margin": 25.0,
    "trim_pad_frames": 5,       # keep a few frames around speech (preserves /s/, /t/ at word edges)
    "min_frames_after_trim": 48,
    "edge_zero_window": 5,      # at first/last N frames, zero-out (-80dB) any silent frame
    "edge_zero_apply": True,

    # Dataset
    "source_emotion": "neutral",
    "target_emotions": ["angry", "happy", "sad"],
    "val_size": 0.10,
    "use_dtw_alignment": True,
    "max_dtw_frames": 420,
    "num_workers": 2,

    # Model dimensions
    "content_dim": 256,
    "aux_dim": 64,
    "speaker_dim": 128,
    "emotion_dim": 64,
    "hidden_dim": 256,
    "dropout": 0.10,

    # Training
    "total_epochs": 250,
    "phase1_epochs": 50,
    "phase2_epochs": 130,
    "phase3_epochs": 70,
    "batch_size": 16,
    "lr_G": 1e-4,
    "lr_D": 5e-5,
    "lr_SER": 1e-4,
    "weight_decay": 1e-5,
    "grad_clip": 5.0,

    # SER pretraining
    "ser_pretrain_epochs": 15,


    # ─── EMOTION-INJECTION FIX hyperparameters (v2) ───────────────────
    "use_emotion_fix": True,
    "lambda_prosody": 3.0,
    "use_online_ser": True,
    "lr_ser_online": 2e-4,
    "online_ser_warmup": 3,
    "use_grl": True,
    "lambda_grl": 1.0,
    "grl_alpha": 1.0,
    "fix_lambda_l1_p2": 4.0,
    "fix_lambda_l1_p3": 3.0,
    "fix_lambda_ser_p2": 3.0,
    "fix_lambda_ser_p3": 5.0,
    "fix_lambda_cycle": 2.0,
    "fix_lambda_adv_p2": 0.3,
    "fix_lambda_adv_p3": 0.6,

    # Checkpoint / eval
    "save_every": 25,
    "visual_eval_every": 25,

    # Kaggle-specific resume behaviour
    "resume": True,                      # Always try to resume from latest checkpoint
    "resume_path": None,                 # If set, force this exact .pt; else auto-find
    "checkpoint_input_dir": str(CHECKPOINT_INPUT_DIR),
}

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in CFG.items()}, indent=2))
print("\nPaths:")
print(f"  CHECKPOINT_INPUT_DIR : {CHECKPOINT_INPUT_DIR}  (exists={CHECKPOINT_INPUT_DIR.exists()})")
print(f"  OUT_DIR              : {OUT_DIR}")
print(f"  CKPT_DIR             : {CKPT_DIR}")

## 3 · Auto-discover the processed feature dataset

Searches `/kaggle/input/` for:
1. A `metadata.csv` (already-extracted dataset folder) → use that folder directly
2. A `subesco_processed_dataset.zip` → extract to `/kaggle/working/`
3. Any `.zip` containing `metadata.csv` inside → extract

In [ ]:
# ============================================================
# BLOCK 3 — Locate / extract processed feature dataset
# ============================================================

def find_processed_dataset():
    """
    Returns (metadata_csv_path, feature_root_dir).
    feature_root_dir is the folder containing mel/, f0/, energy/, voiced/ subfolders.
    """
    # Priority 1: explicit ZIP path
    if EXPLICIT_ZIP_PATH is not None and EXPLICIT_ZIP_PATH.exists():
        return _extract_zip(EXPLICIT_ZIP_PATH)

    # Priority 2: already-extracted dataset in /kaggle/input/
    print("Scanning /kaggle/input/ for metadata.csv ...")
    candidates = list(KAGGLE_INPUT_ROOT.rglob("metadata.csv"))
    # Exclude the raw SubESCO metadata (the processed one has mel_path column)
    valid = []
    for c in candidates:
        try:
            head = pd.read_csv(c, nrows=2)
            if "mel_path" in head.columns:
                valid.append(c)
                print(f"  ✓ {c}  ({len(head.columns)} columns)")
        except Exception:
            pass

    if valid:
        meta_path = valid[0]
        feature_root = meta_path.parent
        print(f"\nUsing already-extracted dataset:")
        print(f"  metadata : {meta_path}")
        print(f"  root     : {feature_root}")
        return meta_path, feature_root

    # Priority 3: any zip in /kaggle/input/
    print("No extracted metadata.csv found; scanning for ZIP files ...")
    zips = list(KAGGLE_INPUT_ROOT.rglob("subesco_processed_dataset.zip"))
    if not zips:
        zips = list(KAGGLE_INPUT_ROOT.rglob("*.zip"))

    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                if any(n.endswith("metadata.csv") for n in names):
                    print(f"  ✓ Found dataset ZIP: {z}")
                    return _extract_zip(z)
        except Exception:
            continue

    raise FileNotFoundError(
        "Could not find processed dataset under /kaggle/input/.\n"
        "Expected either:\n"
        "  (a) a folder containing metadata.csv + mel/ + f0/ + energy/ + voiced/, or\n"
        "  (b) subesco_processed_dataset.zip with the same structure inside."
    )


def _extract_zip(zip_path):
    print(f"Extracting {zip_path}  →  {EXTRACT_DIR}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACT_DIR)

    meta_candidates = list(EXTRACT_DIR.rglob("metadata.csv"))
    if not meta_candidates:
        raise FileNotFoundError(f"No metadata.csv inside extracted {zip_path}")

    meta_path = meta_candidates[0]
    feature_root = meta_path.parent
    print(f"  metadata : {meta_path}")
    print(f"  root     : {feature_root}")
    return meta_path, feature_root


META_PATH, FEATURE_ROOT = find_processed_dataset()
# EXTRACT_DIR is the resolution root for relative paths inside metadata.csv
RESOLVE_ROOTS = [FEATURE_ROOT, EXTRACT_DIR, META_PATH.parent]
print("\nResolve roots:")
for r in RESOLVE_ROOTS:
    print(f"  {r}")

## 4 · Load metadata and resolve feature paths

In [ ]:
# ============================================================
# BLOCK 4 — Load metadata and resolve feature paths
# ============================================================

df = pd.read_csv(META_PATH)
df.columns = [c.strip() for c in df.columns]

print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head())

def pick_col(possible_names, required=True):
    lower_map = {c.lower(): c for c in df.columns}
    for name in possible_names:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    if required:
        raise KeyError(f"Missing required column. Tried: {possible_names}")
    return None

COL_EMOTION = pick_col(["emotion"])
COL_LABEL   = pick_col(["label"], required=False)
COL_SPEAKER = pick_col(["speaker"])
COL_SENT    = pick_col(["sentence", "sent"])
COL_TAKE    = pick_col(["take"])
COL_MEL     = pick_col(["mel_path", "mel", "mel_file"])
COL_F0      = pick_col(["f0_path", "f0", "pitch_path"])
COL_ENERGY  = pick_col(["energy_path", "energy", "energy_file"])
COL_VOICED  = pick_col(["voiced_path", "voiced", "uv_path"])
COL_WAV     = pick_col(["wav_path", "audio_path"], required=False)

df[COL_EMOTION] = df[COL_EMOTION].astype(str).str.lower().str.strip()
df[COL_SPEAKER] = df[COL_SPEAKER].astype(str)

print("\nDetected columns:")
for k, v in {
    "emotion": COL_EMOTION, "label": COL_LABEL, "speaker": COL_SPEAKER,
    "sentence": COL_SENT, "take": COL_TAKE,
    "mel": COL_MEL, "f0": COL_F0, "energy": COL_ENERGY, "voiced": COL_VOICED,
    "wav": COL_WAV,
}.items():
    print(f"  {k:8s} → {v}")

print("\nEmotion counts:")
display(df[COL_EMOTION].value_counts())


def resolve_path(p):
    """Try resolve `p` against all known feature roots."""
    if pd.isna(p):
        return None
    p = str(p)
    cand = Path(p)
    if cand.is_absolute() and cand.exists():
        return cand
    for root in RESOLVE_ROOTS:
        c = root / p
        if c.exists():
            return c
    # Last resort: filename search under the main root
    base = Path(p).name
    for root in RESOLVE_ROOTS:
        hits = list(root.rglob(base))
        if hits:
            return hits[0]
    return None


# Quick verification on first 20 rows
print("\nPath-resolution sanity check (first 20 rows):")
for col in [COL_MEL, COL_F0, COL_ENERGY, COL_VOICED]:
    ok = sum(resolve_path(x) is not None for x in df[col].head(20))
    print(f"  {col}: {ok}/20 resolved")

## 5 · Feature loading, normalization, statistics

### Two-stage silence removal (NEW)

On top of the silence trim done during extraction, every loaded utterance now goes through:

**Stage 1 — voiced-aware cropping**  
A frame is "active" if **either** the energy is within 25 dB of the file's peak **or** the voiced flag is 1 (F0 detected). This union catches:
- Unvoiced consonants (`/s/`, `/t/`, `/k/`) — high energy, voiced=0
- Quiet vowels at word boundaries — low energy, voiced=1

We crop the mel/F0/energy/voiced arrays to the span between the first and last active frame, with `pad_frames=5` of slack so word-initial unvoiced consonants are never clipped.

**Stage 2 — edge zeroing**  
Within only the **first and last `edge_zero_window=5` frames** of the cropped clip, any frame that is *both* unvoiced *and* below the energy threshold is set to `-80 dB` (true silence). The middle is never touched, so unvoiced consonants inside the utterance stay intact.

> ⚠️ A literal `mel × voiced` multiplication would be wrong here: voiced=0 also marks unvoiced consonants (so `/s/` in "sad" would vanish), and multiplying dB values by 0 gives 0 dB (loudest, not silence). The two-stage approach above achieves the goal — silent edges removed — without these failure modes.

In [ ]:
# ============================================================
# BLOCK 5 — Feature loading, silence trimming, normalization, stats
# ============================================================

def ensure_mel_shape(mel):
    mel = np.asarray(mel, dtype=np.float32)
    if mel.ndim != 2:
        raise ValueError(f"Mel must be 2D, got {mel.shape}")
    if mel.shape[0] == CFG["n_mels"]:
        return mel
    if mel.shape[1] == CFG["n_mels"]:
        return mel.T
    raise ValueError(f"Mel has no {CFG['n_mels']} mel dimension: {mel.shape}")

def load_mel_db(path_like):
    path = resolve_path(path_like)
    if path is None:
        raise FileNotFoundError(path_like)
    mel = np.load(path).astype(np.float32)
    mel = ensure_mel_shape(mel)
    mel = np.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)
    mel = np.clip(mel, -80.0, 0.0)
    return mel

def fit_length_1d(x, T):
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) == T:
        return x
    if len(x) > T:
        return x[:T]
    pad = np.zeros(T - len(x), dtype=np.float32)
    return np.concatenate([x, pad], axis=0)

def fit_length_2d(x, T):
    x = np.asarray(x, dtype=np.float32)
    if x.shape[1] == T:
        return x
    if x.shape[1] > T:
        return x[:, :T]
    pad = np.ones((x.shape[0], T - x.shape[1]), dtype=np.float32) * -80.0
    return np.concatenate([x, pad], axis=1)

def load_1d_feature(path_like, expected_len=None, default=None):
    path = resolve_path(path_like)
    if path is None:
        if default is None:
            raise FileNotFoundError(path_like)
        arr = np.asarray(default, dtype=np.float32)
    else:
        arr = np.load(path).astype(np.float32).reshape(-1)
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if expected_len is not None:
        arr = fit_length_1d(arr, expected_len)
    return arr.astype(np.float32)

def derive_energy_from_mel_db(mel_db):
    return mel_db.mean(axis=0).astype(np.float32)

def get_active_region_from_mel_db(mel_db, top_db_margin=None, pad_frames=None, min_active_frames=None):
    """Energy-only trimming (legacy)."""
    if top_db_margin is None:    top_db_margin = CFG["trim_top_db_margin"]
    if pad_frames is None:       pad_frames = CFG["trim_pad_frames"]
    if min_active_frames is None: min_active_frames = CFG["min_frames_after_trim"]

    T = mel_db.shape[1]
    if T <= 1:
        return 0, T
    frame_energy = derive_energy_from_mel_db(mel_db)
    threshold = float(frame_energy.max()) - float(top_db_margin)
    active = frame_energy > threshold
    idx = np.where(active)[0]
    if len(idx) < min_active_frames:
        return 0, T
    start = max(0, int(idx[0]) - int(pad_frames))
    end   = min(T, int(idx[-1]) + int(pad_frames) + 1)
    if end <= start:
        return 0, T
    return start, end


def get_active_region_voiced_aware(mel_db, voiced=None, top_db_margin=None,
                                     pad_frames=None, min_active_frames=None):
    """
    Combined voiced + energy trimming.

    A frame counts as 'active' if EITHER:
      • energy is within `top_db_margin` dB of the file's peak, OR
      • the voiced flag is 1 (F0 was detected here).

    Using OR (union) ensures we capture both unvoiced consonants (high energy,
    voiced=0) AND quiet vowels (low energy, voiced=1). Then we crop to the
    first/last active frame with `pad_frames` of padding.
    """
    if top_db_margin is None:    top_db_margin = CFG["trim_top_db_margin"]
    if pad_frames is None:       pad_frames = CFG["trim_pad_frames"]
    if min_active_frames is None: min_active_frames = CFG["min_frames_after_trim"]

    T = mel_db.shape[1]
    if T <= 1:
        return 0, T

    # Energy mask
    frame_energy = derive_energy_from_mel_db(mel_db)
    e_threshold = float(frame_energy.max()) - float(top_db_margin)
    energy_active = frame_energy > e_threshold

    # Voiced mask (union with energy mask)
    if voiced is not None and len(voiced) == T:
        voiced_active = np.asarray(voiced).reshape(-1) > 0.5
        combined = energy_active | voiced_active
    else:
        combined = energy_active

    idx = np.where(combined)[0]
    if len(idx) < min_active_frames:
        # Not enough active frames detected; keep original to avoid over-cropping
        return 0, T

    start = max(0, int(idx[0]) - int(pad_frames))
    end   = min(T, int(idx[-1]) + int(pad_frames) + 1)
    if end <= start:
        return 0, T
    return start, end


def zero_out_silent_edges(mel_db, voiced, energy=None, window=None, top_db_margin=None):
    """
    Safety net: at the very edges of the (already trimmed) clip, set mel to -80 dB
    for any frame that is BOTH unvoiced AND has energy below threshold.
    Only operates within the first/last `window` frames — the middle of the
    utterance is NEVER touched, so unvoiced consonants in the middle are safe.

    Note: This is the *correct* way to zero out silence in dB-mel space.
    Multiplying mel by 0 would give 0 dB (loudest, not silent), so we use
    direct -80 dB assignment instead.
    """
    if window is None:        window = CFG["edge_zero_window"]
    if top_db_margin is None: top_db_margin = CFG["trim_top_db_margin"]

    T = mel_db.shape[1]
    if T < window * 2 or voiced is None or len(voiced) != T:
        return mel_db

    if energy is None:
        energy = derive_energy_from_mel_db(mel_db)
    e_threshold = float(energy.max()) - float(top_db_margin)

    voiced_arr = np.asarray(voiced).reshape(-1) > 0.5
    mel_out = mel_db.copy()

    # Left edge
    for i in range(window):
        if not voiced_arr[i] and energy[i] < e_threshold:
            mel_out[:, i] = -80.0
    # Right edge
    for i in range(T - window, T):
        if not voiced_arr[i] and energy[i] < e_threshold:
            mel_out[:, i] = -80.0

    return mel_out


def trim_feature_bundle(mel_db, f0_hz=None, energy=None, voiced=None):
    """
    Two-stage silence removal:
      Stage 1 (crop):  voiced-aware boundary detection → crop everything to [start:end]
      Stage 2 (zero):  within the trimmed clip's edges, force residual silence to -80 dB
    """
    if not CFG.get("trim_silence", True):
        return mel_db, f0_hz, energy, voiced, 0, mel_db.shape[1]

    # ── Stage 1: voiced-aware crop ───────────────────────────────────────
    if CFG.get("trim_use_voiced", True) and voiced is not None:
        start, end = get_active_region_voiced_aware(mel_db, voiced=voiced)
    else:
        start, end = get_active_region_from_mel_db(mel_db)

    mel_db = mel_db[:, start:end]
    if f0_hz  is not None: f0_hz  = f0_hz[start:end]
    if energy is not None: energy = energy[start:end]
    if voiced is not None: voiced = voiced[start:end]

    # ── Stage 2: edge zeroing (safety net for residual silence) ─────────
    if CFG.get("edge_zero_apply", True) and voiced is not None:
        mel_db = zero_out_silent_edges(mel_db, voiced, energy=energy)

    return mel_db, f0_hz, energy, voiced, start, end

def normalize_mel(mel_db):
    mel_db = np.clip(mel_db, -80.0, 0.0)
    return ((mel_db + 40.0) / 40.0).astype(np.float32)

def denormalize_mel(mel_norm):
    mel_db = mel_norm * 40.0 - 40.0
    return np.clip(mel_db, -80.0, 0.0).astype(np.float32)

def load_full_features(row):
    mel_db = load_mel_db(row[COL_MEL])
    T = mel_db.shape[1]
    f0_hz  = load_1d_feature(row[COL_F0],     expected_len=T)
    energy = load_1d_feature(row[COL_ENERGY], expected_len=T)
    voiced = load_1d_feature(row[COL_VOICED], expected_len=T)
    voiced = (voiced > 0.5).astype(np.float32)
    mel_db, f0_hz, energy, voiced, ts, te = trim_feature_bundle(
        mel_db, f0_hz=f0_hz, energy=energy, voiced=voiced
    )
    return {
        "mel_db": mel_db.astype(np.float32),
        "f0_hz":  f0_hz.astype(np.float32),
        "energy": energy.astype(np.float32),
        "voiced": voiced.astype(np.float32),
        "trim_start": int(ts), "trim_end": int(te),
    }

def compute_stats(df_train):
    f0_logs, energies, kept = [], [], 0
    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Computing stats"):
        feat = load_full_features(row)
        mel, f0, e = feat["mel_db"], feat["f0_hz"], feat["energy"]
        if mel.shape[1] < CFG["min_frames_after_trim"]:
            continue
        voiced = f0 > 0
        if voiced.any():
            f0_logs.append(np.log(np.maximum(f0[voiced], 1e-6)))
        energies.append(e)
        kept += 1
    f0_logs  = np.concatenate(f0_logs)  if f0_logs  else np.array([math.log(150.0)], dtype=np.float32)
    energies = np.concatenate(energies) if energies else np.array([-50.0], dtype=np.float32)
    return {
        "mel_min": -80.0, "mel_max": 0.0,
        "f0_log_mean": float(f0_logs.mean()),
        "f0_log_std":  float(max(f0_logs.std(), 1e-4)),
        "energy_mean": float(energies.mean()),
        "energy_std":  float(max(energies.std(), 1e-4)),
        "stats_rows_used_after_trim": int(kept),
    }

def normalize_f0(f0_hz, stats):
    f0_hz = np.asarray(f0_hz, dtype=np.float32)
    voiced = f0_hz > 0
    out = np.zeros_like(f0_hz, dtype=np.float32)
    if voiced.any():
        log_f0 = np.log(np.maximum(f0_hz[voiced], 1e-6))
        out[voiced] = (log_f0 - stats["f0_log_mean"]) / (stats["f0_log_std"] + 1e-8)
    return out.astype(np.float32), voiced.astype(np.float32)

def denormalize_f0(f0_norm, voiced, stats):
    f0_norm = np.asarray(f0_norm, dtype=np.float32)
    voiced = np.asarray(voiced) > 0.5
    out = np.zeros_like(f0_norm, dtype=np.float32)
    out[voiced] = np.exp(f0_norm[voiced] * stats["f0_log_std"] + stats["f0_log_mean"])
    return out

def normalize_energy(energy, stats):
    energy = np.asarray(energy, dtype=np.float32)
    return ((energy - stats["energy_mean"]) / (stats["energy_std"] + 1e-8)).astype(np.float32)

def denormalize_energy(energy_norm, stats):
    return energy_norm * stats["energy_std"] + stats["energy_mean"]

print("Feature utilities ready.")

## 6 · Build neutral → emotion paired dataset + compute stats

In [ ]:
# ============================================================
# BLOCK 6 — Build neutral→emotion paired dataset
# ============================================================

source_emo = CFG["source_emotion"]
target_emos = set(CFG["target_emotions"])

df_work = df[df[COL_EMOTION].isin([source_emo] + list(target_emos))].copy().reset_index(drop=True)

emotion_names = sorted(df_work[COL_EMOTION].unique().tolist())
emotion_to_id = {e: i for i, e in enumerate(emotion_names)}
id_to_emotion = {i: e for e, i in emotion_to_id.items()}

speaker_names = sorted(df_work[COL_SPEAKER].unique().tolist())
speaker_to_id = {s: i for i, s in enumerate(speaker_names)}
id_to_speaker = {i: s for s, i in speaker_to_id.items()}

print("Emotion map:", emotion_to_id)
print("Speakers   :", len(speaker_to_id))

def make_key(row, include_take=True):
    parts = [str(row[COL_SPEAKER])]
    if COL_SENT is not None:
        parts.append(str(row[COL_SENT]))
    if include_take and COL_TAKE is not None:
        parts.append(str(row[COL_TAKE]))
    return "||".join(parts)

neutral_df = df_work[df_work[COL_EMOTION] == source_emo].copy()
target_df  = df_work[df_work[COL_EMOTION].isin(target_emos)].copy()

pairs = []
neutral_map = defaultdict(list)
for idx, row in neutral_df.iterrows():
    neutral_map[make_key(row, include_take=True)].append(idx)

for tidx, trow in target_df.iterrows():
    key = make_key(trow, include_take=True)
    if key in neutral_map:
        for nidx in neutral_map[key]:
            pairs.append((nidx, tidx, "strict"))

if len(pairs) < 100 and COL_SENT is not None:
    print("Few strict pairs; trying relaxed speaker+sentence pairing...")
    neutral_map2 = defaultdict(list)
    for idx, row in neutral_df.iterrows():
        neutral_map2[make_key(row, include_take=False)].append(idx)
    used = set((n, t) for n, t, _ in pairs)
    for tidx, trow in target_df.iterrows():
        key = make_key(trow, include_take=False)
        if key in neutral_map2:
            for nidx in neutral_map2[key]:
                if (nidx, tidx) not in used:
                    pairs.append((nidx, tidx, "relaxed"))

if len(pairs) == 0:
    raise RuntimeError(
        "No neutral→target pairs found. Check speaker/sentence/take columns.\n"
        f"Emotions in data: {sorted(df_work[COL_EMOTION].unique().tolist())}"
    )

pairs_df = pd.DataFrame(pairs, columns=["src_idx", "tgt_idx", "pair_type"])
pairs_df["target_emotion"] = [df_work.iloc[t][COL_EMOTION] for t in pairs_df["tgt_idx"]]
pairs_df["speaker"]        = [df_work.iloc[s][COL_SPEAKER] for s in pairs_df["src_idx"]]

print("Total pairs:", len(pairs_df))
display(pairs_df["target_emotion"].value_counts())
display(pairs_df["pair_type"].value_counts())

train_pairs, val_pairs = train_test_split(
    pairs_df, test_size=CFG["val_size"], random_state=SEED,
    stratify=pairs_df["target_emotion"] if pairs_df["target_emotion"].nunique() > 1 else None
)
train_pairs = train_pairs.reset_index(drop=True)
val_pairs   = val_pairs.reset_index(drop=True)
print("Train pairs:", len(train_pairs), " Val pairs:", len(val_pairs))

train_row_indices = sorted(set(train_pairs["src_idx"].tolist() + train_pairs["tgt_idx"].tolist()))
df_train_rows = df_work.iloc[train_row_indices].reset_index(drop=True)

STATS_PATH = OUT_DIR / "stats.json"
if STATS_PATH.exists():
    STATS = json.loads(STATS_PATH.read_text())
    print("Loaded existing stats:", STATS_PATH)
else:
    STATS = compute_stats(df_train_rows)
    STATS_PATH.write_text(json.dumps(STATS, indent=2))
    print("Saved stats:", STATS_PATH)

print(json.dumps(STATS, indent=2))

## 7 · DTW alignment & variable-length PyTorch datasets

In [ ]:
# ============================================================
# BLOCK 7 — DTW alignment and PyTorch datasets (variable-length)
# ============================================================

def align_1d_by_path(src_len, tgt_1d, path_src_to_tgt):
    buckets = [[] for _ in range(src_len)]
    for si, ti in path_src_to_tgt:
        if 0 <= si < src_len and 0 <= ti < len(tgt_1d):
            buckets[si].append(tgt_1d[ti])
    out = np.zeros(src_len, dtype=np.float32)
    for i, vals in enumerate(buckets):
        if vals:
            out[i] = float(np.mean(vals))
        else:
            j = min(max(i, 0), len(tgt_1d) - 1)
            out[i] = tgt_1d[j]
    return out

def align_mel_by_dtw(src_mel_db, tgt_mel_db, tgt_f0, tgt_energy, tgt_voiced, cache_key=None):
    src_T = src_mel_db.shape[1]; tgt_T = tgt_mel_db.shape[1]
    if src_T <= 1 or tgt_T <= 1:
        return (fit_length_2d(tgt_mel_db, src_T), fit_length_1d(tgt_f0, src_T),
                fit_length_1d(tgt_energy, src_T), fit_length_1d(tgt_voiced, src_T))
    if not CFG["use_dtw_alignment"]:
        return (fit_length_2d(tgt_mel_db, src_T), fit_length_1d(tgt_f0, src_T),
                fit_length_1d(tgt_energy, src_T), fit_length_1d(tgt_voiced, src_T))
    if max(src_T, tgt_T) > CFG["max_dtw_frames"]:
        return (fit_length_2d(tgt_mel_db, src_T), fit_length_1d(tgt_f0, src_T),
                fit_length_1d(tgt_energy, src_T), fit_length_1d(tgt_voiced, src_T))

    if cache_key is not None:
        cache_path = CACHE_DIR / f"trimv1_{cache_key}.npz"
        if cache_path.exists():
            z = np.load(cache_path)
            if z["mel"].shape[1] == src_T:
                return z["mel"], z["f0"], z["energy"], z["voiced"]

    X = normalize_mel(src_mel_db); Y = normalize_mel(tgt_mel_db)
    try:
        Dmat, wp = librosa.sequence.dtw(X=X, Y=Y, metric="cosine")
        wp = wp[::-1]
        aligned_mel = np.zeros((CFG["n_mels"], src_T), dtype=np.float32)
        for si in range(src_T):
            tis = [int(ti) for s, ti in wp if int(s) == si]
            if tis:
                aligned_mel[:, si] = tgt_mel_db[:, tis].mean(axis=1)
            else:
                j = min(max(si, 0), tgt_T - 1)
                aligned_mel[:, si] = tgt_mel_db[:, j]
        aligned_f0     = align_1d_by_path(src_T, tgt_f0, wp)
        aligned_energy = align_1d_by_path(src_T, tgt_energy, wp)
        aligned_voiced = align_1d_by_path(src_T, tgt_voiced, wp)
        aligned_voiced = (aligned_voiced > 0.5).astype(np.float32)
    except Exception as e:
        print("DTW failed, using length fit:", e)
        aligned_mel    = fit_length_2d(tgt_mel_db, src_T)
        aligned_f0     = fit_length_1d(tgt_f0,    src_T)
        aligned_energy = fit_length_1d(tgt_energy,src_T)
        aligned_voiced = fit_length_1d(tgt_voiced,src_T)

    if cache_key is not None:
        np.savez_compressed(cache_path, mel=aligned_mel, f0=aligned_f0,
                            energy=aligned_energy, voiced=aligned_voiced)
    return aligned_mel, aligned_f0, aligned_energy, aligned_voiced


class EVCPairedDataset(Dataset):
    def __init__(self, pairs_df, train=True):
        self.pairs_df = pairs_df.reset_index(drop=True)
        self.train = train
    def __len__(self):
        return len(self.pairs_df)
    def __getitem__(self, idx):
        item = self.pairs_df.iloc[idx]
        src_row = df_work.iloc[int(item["src_idx"])]
        tgt_row = df_work.iloc[int(item["tgt_idx"])]
        src = load_full_features(src_row); tgt = load_full_features(tgt_row)
        cache_key = f"pair_{int(item['src_idx'])}_{int(item['tgt_idx'])}"
        tgt_mel_a, tgt_f0_a, tgt_e_a, tgt_v_a = align_mel_by_dtw(
            src["mel_db"], tgt["mel_db"], tgt["f0_hz"], tgt["energy"], tgt["voiced"], cache_key=cache_key)
        T_src = src["mel_db"].shape[1]
        src_mel_norm = normalize_mel(src["mel_db"]); tgt_mel_norm = normalize_mel(tgt_mel_a)
        src_f0_norm, src_v2 = normalize_f0(src["f0_hz"], STATS)
        tgt_f0_norm, tgt_v2 = normalize_f0(tgt_f0_a,     STATS)
        src_e_norm = normalize_energy(src["energy"], STATS)
        tgt_e_norm = normalize_energy(tgt_e_a,       STATS)
        src_aux = np.stack([src_f0_norm, src_e_norm, src_v2], axis=0).astype(np.float32)
        tgt_aux = np.stack([tgt_f0_norm, tgt_e_norm, tgt_v2], axis=0).astype(np.float32)
        return {
            "src_mel": torch.tensor(src_mel_norm, dtype=torch.float32),
            "src_aux": torch.tensor(src_aux,      dtype=torch.float32),
            "tgt_mel": torch.tensor(tgt_mel_norm, dtype=torch.float32),
            "tgt_aux": torch.tensor(tgt_aux,      dtype=torch.float32),
            "src_len": T_src,
            "src_emo": torch.tensor(emotion_to_id[src_row[COL_EMOTION]], dtype=torch.long),
            "tgt_emo": torch.tensor(emotion_to_id[tgt_row[COL_EMOTION]], dtype=torch.long),
            "spk_id":  torch.tensor(speaker_to_id[str(src_row[COL_SPEAKER])], dtype=torch.long),
            "pair_index": torch.tensor(idx, dtype=torch.long),
        }


class SERDataset(Dataset):
    def __init__(self, rows_df, train=True):
        self.rows_df = rows_df.reset_index(drop=True); self.train = train
    def __len__(self):
        return len(self.rows_df)
    def __getitem__(self, idx):
        row = self.rows_df.iloc[idx]
        feat = load_full_features(row)
        mel_norm = normalize_mel(feat["mel_db"])
        label = emotion_to_id[row[COL_EMOTION]]
        return torch.tensor(mel_norm, dtype=torch.float32), torch.tensor(label, dtype=torch.long), mel_norm.shape[1]


def evc_collate_fn(batch):
    max_src_len = max(item["src_len"] for item in batch)
    B = len(batch)
    src_mel_p = torch.full((B, CFG["n_mels"], max_src_len), -1.0, dtype=torch.float32)
    src_aux_p = torch.zeros((B, 3, max_src_len), dtype=torch.float32)
    tgt_mel_p = torch.full((B, CFG["n_mels"], max_src_len), -1.0, dtype=torch.float32)
    tgt_aux_p = torch.zeros((B, 3, max_src_len), dtype=torch.float32)
    masks = torch.zeros((B, max_src_len), dtype=torch.float32)
    s_spk, s_emo, t_emo, p_idx = [], [], [], []
    for i, item in enumerate(batch):
        L = item["src_len"]
        src_mel_p[i, :, :L] = item["src_mel"]
        src_aux_p[i, :, :L] = item["src_aux"]
        tgt_mel_p[i, :, :L] = item["tgt_mel"]
        tgt_aux_p[i, :, :L] = item["tgt_aux"]
        masks[i, :L] = 1.0
        s_spk.append(item["spk_id"]); s_emo.append(item["src_emo"])
        t_emo.append(item["tgt_emo"]); p_idx.append(item["pair_index"])
    return {
        "src_mel": src_mel_p, "src_aux": src_aux_p,
        "tgt_mel": tgt_mel_p, "tgt_aux": tgt_aux_p,
        "mask": masks,
        "spk_id":  torch.stack(s_spk),
        "src_emo": torch.stack(s_emo),
        "tgt_emo": torch.stack(t_emo),
        "pair_index": torch.stack(p_idx),
    }

def ser_collate_fn(batch):
    max_len = max(item[2] for item in batch); B = len(batch)
    mel_p = torch.full((B, CFG["n_mels"], max_len), -1.0, dtype=torch.float32)
    labels, masks = [], torch.zeros((B, max_len), dtype=torch.float32)
    for i, (mel, label, T) in enumerate(batch):
        mel_p[i, :, :T] = mel; masks[i, :T] = 1.0; labels.append(label)
    return mel_p, torch.stack(labels), masks


def apply_output_gating(fake_mel, mask):
    me = mask.unsqueeze(1)
    return fake_mel * me + (-1.0) * (1.0 - me)


train_ds = EVCPairedDataset(train_pairs, train=True)
val_ds   = EVCPairedDataset(val_pairs,   train=False)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True,
                          collate_fn=evc_collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=False,
                          collate_fn=evc_collate_fn)

ser_train_df, ser_val_df = train_test_split(
    df_train_rows, test_size=0.10, random_state=SEED,
    stratify=df_train_rows[COL_EMOTION] if df_train_rows[COL_EMOTION].nunique() > 1 else None)

ser_train_loader = DataLoader(SERDataset(ser_train_df, train=True), batch_size=CFG["batch_size"],
                              shuffle=True, num_workers=CFG["num_workers"], pin_memory=True,
                              drop_last=True, collate_fn=ser_collate_fn)
ser_val_loader   = DataLoader(SERDataset(ser_val_df,   train=False), batch_size=CFG["batch_size"],
                              shuffle=False, num_workers=CFG["num_workers"], pin_memory=True,
                              drop_last=False, collate_fn=ser_collate_fn)

batch = next(iter(train_loader))
print("src_mel:", batch["src_mel"].shape)
print("mask   :", batch["mask"].shape, " valid-ratio:", float(batch["mask"].mean()))
print("Emotion IDs:", emotion_to_id)

## 8 · Sanity visualization before training

In [ ]:
# ============================================================
# BLOCK 8 — Sanity visualization before training
# ============================================================

def plot_mel(ax, mel_db, title):
    img = librosa.display.specshow(
        mel_db, sr=CFG["sample_rate"], hop_length=CFG["hop_length"],
        x_axis="time", y_axis="mel", fmax=CFG["fmax"], ax=ax, cmap="magma")
    ax.set_title(title); ax.set_ylim(0, CFG["fmax"]); return img

def show_pair_from_dataset(ds, idx=0):
    b = ds[idx]
    src_db = denormalize_mel(b["src_mel"].numpy())
    tgt_db = denormalize_mel(b["tgt_mel"].numpy())
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_mel(axes[0], src_db, "Source neutral mel")
    plot_mel(axes[1], tgt_db, f"Target mel: {id_to_emotion[int(b['tgt_emo'])]}")
    plt.tight_layout(); plt.show()

    src_aux, tgt_aux = b["src_aux"].numpy(), b["tgt_aux"].numpy()
    src_f0 = denormalize_f0(src_aux[0], src_aux[2], STATS)
    tgt_f0 = denormalize_f0(tgt_aux[0], tgt_aux[2], STATS)
    src_e  = denormalize_energy(src_aux[1], STATS)
    tgt_e  = denormalize_energy(tgt_aux[1], STATS)
    t = np.arange(len(src_f0)) * CFG["hop_length"] / CFG["sample_rate"]

    plt.figure(figsize=(14, 3))
    plt.plot(t, src_f0, label="source F0"); plt.plot(t, tgt_f0, label="target F0")
    plt.title("F0 contour"); plt.xlabel("Time (s)"); plt.ylabel("Hz")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()

    plt.figure(figsize=(14, 3))
    plt.plot(t, src_e, label="source energy"); plt.plot(t, tgt_e, label="target energy")
    plt.title("Energy contour"); plt.xlabel("Time (s)"); plt.ylabel("dB-like")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()

show_pair_from_dataset(train_ds, idx=0)


# ── Before / after trim visualisation (verifies new voiced-aware trim) ──────
def show_trim_effect(row, title_prefix=""):
    """Load a single row WITHOUT trimming, then WITH trimming, and compare."""
    # Untrimmed
    mel_raw = load_mel_db(row[COL_MEL])
    T_raw = mel_raw.shape[1]
    f0_raw = load_1d_feature(row[COL_F0],     expected_len=T_raw)
    e_raw  = load_1d_feature(row[COL_ENERGY], expected_len=T_raw)
    v_raw  = (load_1d_feature(row[COL_VOICED], expected_len=T_raw) > 0.5).astype(np.float32)

    # Trimmed (current pipeline)
    mel_t, f0_t, e_t, v_t, s, e_idx = trim_feature_bundle(
        mel_raw.copy(), f0_hz=f0_raw.copy(), energy=e_raw.copy(), voiced=v_raw.copy()
    )

    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    plot_mel(axes[0], mel_raw, f"{title_prefix}BEFORE trim — {T_raw} frames")
    # Mark the trim boundaries on the BEFORE plot
    t_axis_raw = np.arange(T_raw) * CFG["hop_length"] / CFG["sample_rate"]
    axes[0].axvline(t_axis_raw[s] if s < T_raw else 0,        color="lime", lw=2, ls="--", alpha=0.8, label=f"crop start ({s})")
    axes[0].axvline(t_axis_raw[min(e_idx, T_raw-1)],          color="red",  lw=2, ls="--", alpha=0.8, label=f"crop end ({e_idx})")
    axes[0].legend(loc="upper right")

    plot_mel(axes[1], mel_t, f"{title_prefix}AFTER trim — {mel_t.shape[1]} frames "
                              f"(removed {T_raw - mel_t.shape[1]} frames of silence)")
    plt.tight_layout(); plt.show()

    print(f"  Original duration : {T_raw * CFG['hop_length'] / CFG['sample_rate']:.2f}s "
          f"({T_raw} frames)")
    print(f"  Trimmed duration  : {mel_t.shape[1] * CFG['hop_length'] / CFG['sample_rate']:.2f}s "
          f"({mel_t.shape[1]} frames)")
    print(f"  Voiced frames in trimmed clip : {int(v_t.sum())}/{len(v_t)} = "
          f"{100*v_t.mean():.1f}%")

# Show on a few random training rows
import random as _r
sample_rows = df_train_rows.sample(min(3, len(df_train_rows)), random_state=SEED)
for i, (_, row) in enumerate(sample_rows.iterrows()):
    print(f"\n── Sample {i+1}: {row[COL_SPEAKER]} | {row[COL_EMOTION]} ──")
    show_trim_effect(row, title_prefix=f"[{row[COL_EMOTION]}] ")

## 9 · Model definitions — Generator, Discriminator, SER classifier

In [ ]:
# ============================================================
# BLOCK 9 — Model definitions
# ============================================================

class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, k=5, p=None, dropout=0.0):
        super().__init__()
        if p is None: p = k // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=p),
            nn.InstanceNorm1d(out_ch, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
        )
    def forward(self, x): return self.net(x)

class ContentEncoder(nn.Module):
    def __init__(self, n_mels=128, content_dim=256, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock1D(n_mels, 128, k=7, dropout=dropout),
            ConvBlock1D(128, 192, k=5, dropout=dropout),
            ConvBlock1D(192, content_dim, k=5, dropout=dropout),
            ConvBlock1D(content_dim, content_dim, k=3, dropout=dropout),
        )
    def forward(self, mel): return self.net(mel)

class AuxEncoder(nn.Module):
    def __init__(self, aux_in=3, aux_dim=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock1D(aux_in, 32, k=5, dropout=dropout),
            ConvBlock1D(32, aux_dim, k=5, dropout=dropout),
        )
    def forward(self, aux): return self.net(aux)

class Decoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, n_mels=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock1D(in_dim, hidden_dim, k=5, dropout=dropout),
            ConvBlock1D(hidden_dim, hidden_dim, k=5, dropout=dropout),
            ConvBlock1D(hidden_dim, hidden_dim, k=3, dropout=dropout),
            nn.Conv1d(hidden_dim, n_mels, kernel_size=1),
            nn.Tanh(),
        )
    def forward(self, x): return self.net(x)


# ─── Gradient Reversal Layer (for emotion disentanglement on content) ───────
class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

def grad_reverse(x, alpha=1.0):
    return _GradReverse.apply(x, alpha)

class EmotionFromContent(nn.Module):
    """Predict emotion from time-pooled content; GRL makes content emotion-agnostic."""
    def __init__(self, content_dim, n_emotions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(content_dim, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, n_emotions),
        )
    def forward(self, content_feat, alpha=1.0):
        pooled = content_feat.mean(dim=2)
        rev = grad_reverse(pooled, alpha)
        return self.net(rev)

class EVCGenerator(nn.Module):
    def __init__(self, n_speakers, n_emotions):
        super().__init__()
        self.content_encoder = ContentEncoder(CFG["n_mels"], CFG["content_dim"], CFG["dropout"])
        self.aux_encoder     = AuxEncoder(3, CFG["aux_dim"], CFG["dropout"])
        self.spk_emb = nn.Embedding(n_speakers, CFG["speaker_dim"])
        self.emo_emb = nn.Embedding(n_emotions, CFG["emotion_dim"])
        in_dim = CFG["content_dim"] + CFG["aux_dim"] + CFG["speaker_dim"] + CFG["emotion_dim"]
        self.decoder = Decoder(in_dim, CFG["hidden_dim"], CFG["n_mels"], CFG["dropout"])
    def forward(self, src_mel, src_aux, spk_id, tgt_emo, return_content=False):
        B, _, T = src_mel.shape
        content = self.content_encoder(src_mel)
        aux     = self.aux_encoder(src_aux)
        spk     = self.spk_emb(spk_id).unsqueeze(-1).expand(-1, -1, T)
        emo     = self.emo_emb(tgt_emo).unsqueeze(-1).expand(-1, -1, T)
        x = torch.cat([content, aux, spk, emo], dim=1)
        out = self.decoder(x)
        if return_content:
            return out, content
        return out

class MelDiscriminator(nn.Module):
    def __init__(self, n_emotions):
        super().__init__()
        self.emo_emb = nn.Embedding(n_emotions, 16)
        in_ch = CFG["n_mels"] + 16
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 128, 5, padding=2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(128, 128, 5, padding=2),   nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(128, 64,  5, padding=2),   nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(64, 1, 1),
        )
    def forward(self, mel, emo_id):
        B, _, T = mel.shape
        emo = self.emo_emb(emo_id).unsqueeze(-1).expand(-1, -1, T)
        x = torch.cat([mel, emo], dim=1)
        return self.net(x).mean(dim=[1, 2])

class SERClassifier(nn.Module):
    def __init__(self, n_emotions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(CFG["n_mels"], 96, 5, padding=2), nn.BatchNorm1d(96), nn.ReLU(inplace=True), nn.MaxPool1d(2),
            nn.Conv1d(96, 160, 5, padding=2), nn.BatchNorm1d(160), nn.ReLU(inplace=True), nn.MaxPool1d(2),
            nn.Conv1d(160, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(256, n_emotions)
    def forward(self, mel):
        return self.fc(self.net(mel).squeeze(-1))

n_speakers = len(speaker_to_id)
n_emotions = len(emotion_to_id)

G   = EVCGenerator(n_speakers, n_emotions).to(DEVICE)
D   = MelDiscriminator(n_emotions).to(DEVICE)
SER = SERClassifier(n_emotions).to(DEVICE)

# v2 emotion-fix modules
EMO_FROM_CONTENT = EmotionFromContent(CFG["content_dim"], n_emotions).to(DEVICE)
SER_ONLINE = SERClassifier(n_emotions).to(DEVICE)

def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
print("G   params:", f"{count_params(G):,}")
print("D   params:", f"{count_params(D):,}")
print("SER params:", f"{count_params(SER):,}")
print("EMO_FROM_CONTENT params:", f"{count_params(EMO_FROM_CONTENT):,}")
print("SER_ONLINE params:", f"{count_params(SER_ONLINE):,}")

## 10 · Checkpoint utilities — auto-discovery from input dataset

**This is the new Kaggle resume logic.** It looks for `.pt` files in:
1. `/kaggle/input/datasets/yousufasgormumin57/checkpoint-a-i-r/` (your uploaded checkpoint dataset)
2. `/kaggle/working/evc_clean_250_output/checkpoints/` (this-session output)

It picks the most recent by epoch number (preferring `evc_epoch_NNN.pt`, then `evc_latest.pt`, then any `*.pt`).

In [ ]:
# ============================================================
# BLOCK 10 — Checkpoint discovery & I/O (Kaggle) — FIXED
# ============================================================

import re as _re

def _epoch_from_name(p: Path) -> int:
    m = _re.search(r"epoch[_-]?(\d+)", p.stem, _re.IGNORECASE)
    return int(m.group(1)) if m else -1

def list_checkpoints(directory: Path):
    if not directory.exists():
        return []
    cks = sorted(directory.glob("*.pt"))
    if not cks:
        cks = sorted(directory.rglob("*.pt"))
    return cks

def find_latest_checkpoint():
    """
    Auto-discover the best resume checkpoint.
    Only returns checkpoints that contain a 'G' key (full EVC model).
    """
    if CFG.get("resume_path"):
        p = Path(CFG["resume_path"])
        return p if p.exists() else None

    candidates = []
    for d in [CHECKPOINT_INPUT_DIR, CKPT_DIR]:
        for c in list_checkpoints(d):
            # Skip known non-EVC files
            if c.name in ("ser_classifier_best.pt", "ser_classifier_latest.pt"):
                continue
            candidates.append(c)

    if not candidates:
        return None

    scored = []
    for c in candidates:
        ep_name = _epoch_from_name(c)
        try:
            head = torch.load(c, map_location="cpu", weights_only=False)
            # Must be a dict and contain the 'G' key (full EVC checkpoint)
            if not isinstance(head, dict) or "G" not in head:
                continue
            ep_meta = head.get("epoch", -1)
        except Exception:
            continue
        ep = max(ep_name, ep_meta)
        scored.append((ep, c.stat().st_mtime, c))

    if not scored:
        return None
    scored.sort(key=lambda x: (x[0], x[1]))
    return scored[-1][2]


def save_checkpoint(epoch, tag="latest", extra=None):
    ckpt = {
        "epoch": epoch,
        "G":   G.state_dict(),
        "D":   D.state_dict(),
        "SER": SER.state_dict(),
        "EMO_FROM_CONTENT": EMO_FROM_CONTENT.state_dict() if 'EMO_FROM_CONTENT' in globals() else None,
        "SER_ONLINE":       SER_ONLINE.state_dict()       if 'SER_ONLINE'       in globals() else None,
        "opt_G_state":  opt_G.state_dict()  if 'opt_G'  in globals() else None,
        "opt_D_state":  opt_D.state_dict()  if 'opt_D'  in globals() else None,
        "opt_grl_state":        opt_grl.state_dict()        if 'opt_grl'        in globals() else None,
        "opt_ser_online_state": opt_ser_online.state_dict() if 'opt_ser_online' in globals() else None,
        "history":      history             if 'history' in globals() else [],
        "emotion_to_id": emotion_to_id,
        "speaker_to_id": speaker_to_id,
        "stats": STATS,
        "cfg": CFG,
    }
    if extra:
        ckpt.update(extra)
    path = CKPT_DIR / f"evc_{tag}.pt"
    torch.save(ckpt, path)
    return path


def load_checkpoint(path, load_optims=True):
    """
    Loads model weights from `path`.
    Returns epoch number, or 0 if loading fails (caller should handle).
    """
    global history
    try:
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    except Exception as e:
        print(f"Failed to load checkpoint {path}: {e}")
        return 0

    if "G" not in ckpt:
        print(f"Checkpoint {path} does not contain 'G' key. Skipping.")
        return 0

    G.load_state_dict(ckpt["G"])
    if "D"   in ckpt: D.load_state_dict(ckpt["D"])
    if "SER" in ckpt: SER.load_state_dict(ckpt["SER"])

    # v2 modules
    if "EMO_FROM_CONTENT" in globals() and ckpt.get("EMO_FROM_CONTENT") is not None:
        try: EMO_FROM_CONTENT.load_state_dict(ckpt["EMO_FROM_CONTENT"])
        except Exception as e: print(f"  (skip EMO_FROM_CONTENT: {e})")
    if "SER_ONLINE" in globals() and ckpt.get("SER_ONLINE") is not None:
        try: SER_ONLINE.load_state_dict(ckpt["SER_ONLINE"])
        except Exception as e: print(f"  (skip SER_ONLINE: {e})")
    elif "SER_ONLINE" in globals():
        try: SER_ONLINE.load_state_dict(SER.state_dict()); print("  (SER_ONLINE warm-started from SER)")
        except Exception: pass

    if load_optims and "opt_G" in globals() and ckpt.get("opt_G_state") is not None:
        try: opt_G.load_state_dict(ckpt["opt_G_state"])
        except Exception as e: print(f"  (skip opt_G state: {e})")
    if load_optims and "opt_D" in globals() and ckpt.get("opt_D_state") is not None:
        try: opt_D.load_state_dict(ckpt["opt_D_state"])
        except Exception as e: print(f"  (skip opt_D state: {e})")
    if load_optims and "opt_grl" in globals() and ckpt.get("opt_grl_state") is not None:
        try: opt_grl.load_state_dict(ckpt["opt_grl_state"])
        except Exception as e: print(f"  (skip opt_grl state: {e})")
    if load_optims and "opt_ser_online" in globals() and ckpt.get("opt_ser_online_state") is not None:
        try: opt_ser_online.load_state_dict(ckpt["opt_ser_online_state"])
        except Exception as e: print(f"  (skip opt_ser_online state: {e})")

    history = ckpt.get("history", [])
    ep = int(ckpt.get("epoch", 0))
    print(f"Loaded checkpoint: {path}")
    print(f"  epoch in ckpt = {ep}")
    return ep


# Quick visibility
print(f"\nCheckpoint input dir: {CHECKPOINT_INPUT_DIR}")
print(f"  exists: {CHECKPOINT_INPUT_DIR.exists()}")
if CHECKPOINT_INPUT_DIR.exists():
    found = list_checkpoints(CHECKPOINT_INPUT_DIR)
    print(f"  .pt files found: {len(found)}")
    for c in found[:10]:
        print(f"    - {c.name}  ({c.stat().st_size/1e6:.1f} MB)")

print(f"\nLocal checkpoint dir: {CKPT_DIR}")
print(f"  .pt files found: {len(list_checkpoints(CKPT_DIR))}")

latest_ckpt = find_latest_checkpoint()
print(f"\n→ Latest valid EVC checkpoint auto-selected: {latest_ckpt}")

## 11 · SER classifier — load from input checkpoint OR pretrain

Logic:
1. If a checkpoint with `SER` state exists in the input dataset → load and skip pretraining
2. Else if `ser_classifier_best.pt` is in the working dir → load
3. Else → pretrain for `CFG["ser_pretrain_epochs"]` epochs

In [ ]:
# # ============================================================
# # BLOCK 11 — Load or pretrain SER classifier, then freeze
# # ============================================================

# SER_CKPT = CKPT_DIR / "ser_classifier_best.pt"

# def evaluate_ser(model, loader):
#     model.eval()
#     total, correct, loss_sum = 0, 0, 0.0
#     with torch.no_grad():
#         for mel, y, _ in loader:
#             mel, y = mel.to(DEVICE), y.to(DEVICE)
#             logits = model(mel)
#             loss = F.cross_entropy(logits, y)
#             pred = logits.argmax(dim=1)
#             total += y.numel(); correct += (pred == y).sum().item()
#             loss_sum += loss.item() * y.numel()
#     return loss_sum / max(1, total), correct / max(1, total)

# ser_loaded = False

# # Option A: load SER weights from a full EVC checkpoint in the input dataset
# latest = find_latest_checkpoint()
# if latest is not None:
#     try:
#         ck = torch.load(latest, map_location=DEVICE, weights_only=False)
#         if isinstance(ck, dict) and "SER" in ck:
#             SER.load_state_dict(ck["SER"])
#             print(f"✓ Loaded SER weights from EVC checkpoint: {latest}")
#             ser_loaded = True
#     except Exception as e:
#         print(f"Could not read SER from {latest}: {e}")

# # Option B: separate ser_classifier_best.pt in working dir
# if not ser_loaded and SER_CKPT.exists():
#     SER.load_state_dict(torch.load(SER_CKPT, map_location=DEVICE, weights_only=False))
#     print(f"✓ Loaded SER from {SER_CKPT}")
#     ser_loaded = True

# # Option C: pretrain
# if not ser_loaded:
#     print("No SER checkpoint available — pretraining now ...")
#     opt_ser = torch.optim.AdamW(SER.parameters(), lr=CFG["lr_SER"], weight_decay=CFG["weight_decay"])
#     best_acc = -1.0
#     for ep in range(1, CFG["ser_pretrain_epochs"] + 1):
#         SER.train(); running, seen = 0.0, 0
#         for mel, y, _ in tqdm(ser_train_loader, desc=f"SER ep {ep}/{CFG['ser_pretrain_epochs']}"):
#             mel, y = mel.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
#             opt_ser.zero_grad(set_to_none=True)
#             logits = SER(mel)
#             loss = F.cross_entropy(logits, y)
#             loss.backward()
#             nn.utils.clip_grad_norm_(SER.parameters(), CFG["grad_clip"])
#             opt_ser.step()
#             running += loss.item() * y.numel(); seen += y.numel()
#         vloss, vacc = evaluate_ser(SER, ser_val_loader)
#         print(f"  SER ep {ep:03d}  train_loss={running/max(1,seen):.4f}  val_loss={vloss:.4f}  val_acc={vacc:.3f}")
#         if vacc > best_acc:
#             best_acc = vacc
#             torch.save(SER.state_dict(), SER_CKPT)
#             print(f"  ✓ saved SER best (acc={best_acc:.3f})")

# # Freeze SER
# SER.eval()
# for p in SER.parameters():
#     p.requires_grad_(False)
# print("\nSER frozen and ready.")

In [ ]:
# ============================================================
# BLOCK 11 — Load or pretrain SER classifier, then freeze
# ============================================================

import logging, io, contextlib
SER_CKPT = CKPT_DIR / "ser_classifier_best.pt"

# Silence all warnings (FutureWarning, UserWarning, librosa, torch ...) for this block
warnings.filterwarnings("ignore")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("librosa").setLevel(logging.ERROR)


def _silent_torch_load(path):
    """torch.load that drops stderr noise (deprecation / pickle / weights_only warnings)."""
    buf = io.StringIO()
    with contextlib.redirect_stderr(buf), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return torch.load(path, map_location=DEVICE, weights_only=False)


def evaluate_ser(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for mel, y, _ in loader:
            mel, y = mel.to(DEVICE), y.to(DEVICE)
            logits = model(mel)
            loss = F.cross_entropy(logits, y)
            pred = logits.argmax(dim=1)
            total += y.numel(); correct += (pred == y).sum().item()
            loss_sum += loss.item() * y.numel()
    return loss_sum / max(1, total), correct / max(1, total)


ser_loaded = False

# Option A: load SER weights from a full EVC checkpoint in the input dataset
latest = find_latest_checkpoint()
if latest is not None:
    try:
        ck = _silent_torch_load(latest)
        if isinstance(ck, dict) and "SER" in ck:
            SER.load_state_dict(ck["SER"])
            ser_loaded = True
    except Exception:
        pass

# Option B: separate ser_classifier_best.pt in working dir
if not ser_loaded and SER_CKPT.exists():
    try:
        SER.load_state_dict(_silent_torch_load(SER_CKPT))
        ser_loaded = True
    except Exception:
        pass

# Option C: pretrain — SHOW ONLY THE TRAINING STATE
if not ser_loaded:
    opt_ser = torch.optim.AdamW(SER.parameters(), lr=CFG["lr_SER"], weight_decay=CFG["weight_decay"])
    best_acc = -1.0

    # Header
    print(f"{'epoch':>5} | {'train_loss':>10} | {'val_loss':>9} | {'val_acc':>7} | best")
    print("─" * 55)

    for ep in range(1, CFG["ser_pretrain_epochs"] + 1):
        SER.train(); running, seen = 0.0, 0
        # Suppress warnings inside the training loop too
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for mel, y, _ in tqdm(ser_train_loader, desc=f"SER ep {ep:03d}", leave=False):
                mel, y = mel.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt_ser.zero_grad(set_to_none=True)
                logits = SER(mel)
                loss = F.cross_entropy(logits, y)
                loss.backward()
                nn.utils.clip_grad_norm_(SER.parameters(), CFG["grad_clip"])
                opt_ser.step()
                running += loss.item() * y.numel(); seen += y.numel()

            vloss, vacc = evaluate_ser(SER, ser_val_loader)

        is_best = vacc > best_acc
        marker = " ✓" if is_best else ""
        if is_best:
            best_acc = vacc
            torch.save(SER.state_dict(), SER_CKPT)

        print(f"{ep:>5d} | {running/max(1,seen):>10.4f} | {vloss:>9.4f} | {vacc:>7.3f} |{marker}")

    print("─" * 55)
    print(f"  best val_acc: {best_acc:.3f}")

# Freeze the REFERENCE SER (fixed yardstick + half the SER loss)
SER.eval()
for p in SER.parameters():
    p.requires_grad_(False)

# Warm-start the ONLINE SER from the reference SER, then it keeps adapting (NOT frozen).
if CFG.get("use_online_ser", False) and "SER_ONLINE" in globals():
    try:
        SER_ONLINE.load_state_dict(SER.state_dict())
        print("Online SER warm-started from reference SER.")
    except Exception as e:
        print(f"Online SER warm-start skipped: {e}")

## 12 · Phase schedule, optimizers, masked losses, evaluation helpers

In [ ]:
# ============================================================
# BLOCK 12 — Dynamic 3-phase schedule + helpers
# ============================================================

def lerp(a, b, t):
    return a + (b - a) * float(np.clip(t, 0.0, 1.0))

def get_phase(epoch):
    fix = CFG.get("use_emotion_fix", False)
    if epoch <= CFG["phase1_epochs"]:
        return {"name": "PHASE 1 — reconstruction", "mode": "reconstruct",
                "lambda_l1": 20.0, "lambda_content": 0.0, "lambda_cycle": 0.0,
                "lambda_energy": 2.0, "lambda_ser": 0.0, "lambda_adv": 0.0,
                "lambda_prosody": 0.0, "lambda_grl": 0.0,
                "lr_G_scale": 1.0, "lr_D_scale": 0.0, "instance_noise": 0.0}
    if epoch <= CFG["phase1_epochs"] + CFG["phase2_epochs"]:
        t = (epoch - CFG["phase1_epochs"]) / CFG["phase2_epochs"]
        if fix:
            return {"name": "PHASE 2 — emotion injection (v2 fix)", "mode": "convert",
                    "lambda_l1": lerp(CFG["fix_lambda_l1_p2"], CFG["fix_lambda_l1_p2"]*0.9, t),
                    "lambda_content": 6.0, "lambda_cycle": CFG["fix_lambda_cycle"],
                    "lambda_energy": 2.0,
                    "lambda_ser": lerp(CFG["fix_lambda_ser_p2"]*0.5, CFG["fix_lambda_ser_p2"], t),
                    "lambda_adv": lerp(CFG["fix_lambda_adv_p2"]*0.5, CFG["fix_lambda_adv_p2"], t),
                    "lambda_prosody": CFG["lambda_prosody"],
                    "lambda_grl": CFG["lambda_grl"] if CFG.get("use_grl", False) else 0.0,
                    "lr_G_scale": 1.0, "lr_D_scale": 1.0, "instance_noise": lerp(0.03, 0.01, t)}
        return {"name": "PHASE 2 — controlled emotion conversion", "mode": "convert",
                "lambda_l1": lerp(10.0, 8.0, t), "lambda_content": 8.0, "lambda_cycle": 5.0,
                "lambda_energy": 2.0, "lambda_ser": lerp(0.5, 1.0, t), "lambda_adv": lerp(0.05, 0.20, t),
                "lambda_prosody": 0.0, "lambda_grl": 0.0,
                "lr_G_scale": 1.0, "lr_D_scale": 1.0, "instance_noise": lerp(0.03, 0.01, t)}
    t = (epoch - CFG["phase1_epochs"] - CFG["phase2_epochs"]) / CFG["phase3_epochs"]
    if fix:
        return {"name": "PHASE 3 — emotion sharpening (v2 fix)", "mode": "convert",
                "lambda_l1": lerp(CFG["fix_lambda_l1_p3"], CFG["fix_lambda_l1_p3"]*0.8, t),
                "lambda_content": 6.0, "lambda_cycle": CFG["fix_lambda_cycle"],
                "lambda_energy": 2.0,
                "lambda_ser": lerp(CFG["fix_lambda_ser_p3"]*0.8, CFG["fix_lambda_ser_p3"], t),
                "lambda_adv": lerp(CFG["fix_lambda_adv_p3"]*0.7, CFG["fix_lambda_adv_p3"], t),
                "lambda_prosody": CFG["lambda_prosody"],
                "lambda_grl": CFG["lambda_grl"] if CFG.get("use_grl", False) else 0.0,
                "lr_G_scale": lerp(1.0, 0.5, t), "lr_D_scale": 1.0, "instance_noise": lerp(0.01, 0.0, t)}
    return {"name": "PHASE 3 — mild sharpening", "mode": "convert",
            "lambda_l1": lerp(8.0, 6.0, t), "lambda_content": 8.0, "lambda_cycle": 5.0,
            "lambda_energy": 2.0, "lambda_ser": lerp(1.0, 1.5, t), "lambda_adv": lerp(0.20, 0.50, t),
            "lambda_prosody": 0.0, "lambda_grl": 0.0,
            "lr_G_scale": lerp(1.0, 0.5, t), "lr_D_scale": 1.0, "instance_noise": lerp(0.01, 0.0, t)}

opt_G = torch.optim.AdamW(G.parameters(), lr=CFG["lr_G"], betas=(0.5, 0.9), weight_decay=CFG["weight_decay"])
opt_D = torch.optim.AdamW(D.parameters(), lr=CFG["lr_D"], betas=(0.5, 0.9), weight_decay=CFG["weight_decay"])

# v2 fix optimizers
opt_grl = torch.optim.AdamW(EMO_FROM_CONTENT.parameters(), lr=CFG["lr_G"], betas=(0.5, 0.9),
                            weight_decay=CFG["weight_decay"])
opt_ser_online = torch.optim.AdamW(SER_ONLINE.parameters(), lr=CFG["lr_ser_online"], betas=(0.5, 0.9),
                                   weight_decay=CFG["weight_decay"])

history = []
content_teacher = None

def set_lr(opt, base_lr, scale):
    for pg in opt.param_groups:
        pg["lr"] = base_lr * scale

def add_instance_noise(x, std):
    return x if std <= 0 else x + torch.randn_like(x) * std

def mel_frame_energy(mel_norm):
    return mel_norm.mean(dim=1)

# ─── Prosody-statistics emotion loss (v2 fix) ──────────────────────────────
def _masked_mean(x, mask):
    return (x * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)

def _masked_std(x, mask):
    m = _masked_mean(x, mask).unsqueeze(1)
    var = (((x - m) ** 2) * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
    return torch.sqrt(var + 1e-8)

def prosody_loss(fake_mel, tgt_aux, mask):
    """Match generated energy mean/std/dynamics to the target's. Signal-level → unfakeable."""
    gen_energy = fake_mel.mean(dim=1)               # (B,T)
    tgt_energy = tgt_aux[:, 1, :]                    # (B,T) target energy(norm)
    g_mean = _masked_mean(gen_energy, mask); t_mean = _masked_mean(tgt_energy, mask)
    g_std  = _masked_std(gen_energy, mask);  t_std  = _masked_std(tgt_energy, mask)
    gd = torch.abs(gen_energy[:, 1:] - gen_energy[:, :-1])
    td = torch.abs(tgt_energy[:, 1:] - tgt_energy[:, :-1])
    g_dyn = _masked_mean(gd, mask[:, 1:]); t_dyn = _masked_mean(td, mask[:, 1:])
    return (F.l1_loss(g_mean, t_mean) + F.l1_loss(g_std, t_std) + F.l1_loss(g_dyn, t_dyn))

def masked_l1_loss(pred, target, mask):
    mask = mask[:, None, :].to(pred.device)
    loss = torch.abs(pred - target) * mask
    denom = mask.sum() * pred.shape[1] + 1e-8
    return loss.sum() / denom

def masked_l1_loss_1d(pred, target, mask):
    mask = mask.to(pred.device)
    loss = torch.abs(pred - target) * mask
    return loss.sum() / (mask.sum() + 1e-8)

def make_content_teacher():
    teacher = copy.deepcopy(G.content_encoder).to(DEVICE)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    return teacher

@torch.no_grad()
def evaluate_epoch(epoch, max_batches=None):
    G.eval(); D.eval()
    phase = get_phase(epoch)
    totals = defaultdict(float); count = 0
    correct_ser = 0; ser_total = 0

    for bi, batch in enumerate(val_loader):
        if max_batches is not None and bi >= max_batches: break
        src_mel = batch["src_mel"].to(DEVICE); src_aux = batch["src_aux"].to(DEVICE)
        tgt_mel = batch["tgt_mel"].to(DEVICE); mask = batch["mask"].to(DEVICE)
        src_emo = batch["src_emo"].to(DEVICE); tgt_emo = batch["tgt_emo"].to(DEVICE)
        spk_id  = batch["spk_id"].to(DEVICE)

        desired_mel, desired_emo = (src_mel, src_emo) if phase["mode"] == "reconstruct" else (tgt_mel, tgt_emo)
        fake = G(src_mel, src_aux, spk_id, desired_emo)
        fake = apply_output_gating(fake, mask)

        loss_l1     = masked_l1_loss(fake, desired_mel, mask)
        loss_energy = masked_l1_loss_1d(mel_frame_energy(fake), mel_frame_energy(desired_mel), mask)
        loss_content = torch.tensor(0.0, device=DEVICE)
        if content_teacher is not None and phase["lambda_content"] > 0:
            fake_c = content_teacher(fake); src_c = content_teacher(src_mel)
            loss_content = masked_l1_loss(fake_c, src_c, mask)
        loss_cycle = torch.tensor(0.0, device=DEVICE)
        if phase["lambda_cycle"] > 0:
            cyc = G(fake, src_aux, spk_id, src_emo); cyc = apply_output_gating(cyc, mask)
            loss_cycle = masked_l1_loss(cyc, src_mel, mask)
        loss_ser = torch.tensor(0.0, device=DEVICE)
        if phase["lambda_ser"] > 0:
            logits = SER(fake)
            loss_ser = F.cross_entropy(logits, desired_emo)
            correct_ser += (logits.argmax(1) == desired_emo).sum().item(); ser_total += desired_emo.numel()
        loss_adv = torch.tensor(0.0, device=DEVICE)
        if phase["lambda_adv"] > 0:
            pf = D(fake, desired_emo); loss_adv = F.mse_loss(pf, torch.ones_like(pf))

        total = (phase["lambda_l1"]*loss_l1 + phase["lambda_energy"]*loss_energy +
                 phase["lambda_content"]*loss_content + phase["lambda_cycle"]*loss_cycle +
                 phase["lambda_ser"]*loss_ser + phase["lambda_adv"]*loss_adv)

        bs = src_mel.size(0); count += bs
        for k, v in {"val_total": total, "val_l1": loss_l1, "val_energy": loss_energy,
                     "val_content": loss_content, "val_cycle": loss_cycle,
                     "val_ser": loss_ser, "val_adv": loss_adv}.items():
            totals[k] += float(v.item()) * bs

    out = {k: v / max(1, count) for k, v in totals.items()}
    out["val_ser_acc"] = correct_ser / max(1, ser_total) if ser_total > 0 else 0.0
    return out

def plot_history():
    if not history:
        print("No history yet."); return
    h = pd.DataFrame(history)
    display(h.tail())
    for grp, title in [
        (["train_total","val_total"], "Total loss"),
        (["train_l1","val_l1"], "Mel L1"),
        (["train_content","val_content"], "Content loss"),
        (["train_cycle","val_cycle"], "Cycle loss"),
        (["train_ser","val_ser"], "SER loss"),
        (["train_advG","val_adv"], "Adversarial loss"),
        (["train_prosody"], "Prosody-statistics loss (v2)"),
        (["train_grl"], "GRL emotion-disentanglement loss (v2)"),
        (["train_ser_online"], "Online-SER training loss (v2)"),
        (["val_ser_acc"], "Generated SER target accuracy"),
    ]:
        cols = [c for c in grp if c in h.columns]
        if not cols: continue
        plt.figure(figsize=(10, 4))
        for c in cols: plt.plot(h["epoch"], h[c], label=c)
        plt.title(title); plt.xlabel("Epoch"); plt.grid(True, alpha=0.3); plt.legend(); plt.show()

# Show what each phase looks like
phase_table = pd.DataFrame([{"epoch": e, **get_phase(e)} for e in [1, 25, 50, 51, 100, 150, 180, 181, 200, 225, 250]])
display(phase_table)
print("Helpers ready.")

## 13 · 🔄 Always-on checkpoint resume (before training OR inference)

**This cell runs every time.** It auto-finds the latest checkpoint and loads:
- Generator weights → `G`
- Discriminator weights → `D`  
- SER weights → `SER`
- Optimizer states (only if you continue training)
- Training history → `history`

If no checkpoint exists, training starts fresh from epoch 1.

In [ ]:
# ============================================================
# BLOCK 13 — Always-on resume from latest checkpoint (safe)
# ============================================================

start_epoch = 1
resumed_from = None

if CFG["resume"]:
    ckpt_path = find_latest_checkpoint()
    if ckpt_path is not None:
        loaded_epoch = load_checkpoint(ckpt_path, load_optims=True)
        if loaded_epoch > 0:
            start_epoch = loaded_epoch + 1
            resumed_from = ckpt_path
            print(f"\n✅ Resumed from epoch {loaded_epoch} — next epoch = {start_epoch}")
        else:
            print("\n⚠️ Checkpoint loading failed — starting from scratch.")
    else:
        print("\n🆕 No valid EVC checkpoint found — starting from scratch (epoch 1).")
else:
    print("\nResume disabled in CFG. Starting from scratch.")

# Rebuild content teacher if we resumed past Phase 1
if start_epoch > CFG["phase1_epochs"] + 1:
    content_teacher = make_content_teacher()
    print("Content teacher restored from current G.content_encoder.")

print(f"\n→ start_epoch = {start_epoch}")
print(f"→ resumed_from = {resumed_from}")

## 14 · ⚡ Inference-only fast path — SKIP if you want to train

> **Run this cell only if you want to do inference WITHOUT training** (e.g., mid-experiment, you want to listen to the current model's output and then stop).
>
> If you intend to train, **skip this cell** and go straight to Cell 15.

After running this, jump down to Cell 16 (visualization + audio) and Cell 17 (batch metrics).

In [ ]:
# # ============================================================
# # BLOCK 14 — Inference-only quick-load (OPTIONAL)
# # ============================================================

# # Only re-load if there's a checkpoint AND user hasn't run training yet
# ck = find_latest_checkpoint()
# if ck is None:
#     print("⚠ No checkpoint available — cannot do inference without trained weights.")
#     print("  → Either run training (Cell 15) first, or upload a checkpoint dataset.")
# else:
#     print(f"Loading latest checkpoint for inference: {ck}")
#     load_checkpoint(ck, load_optims=False)
#     G.eval(); D.eval(); SER.eval()
#     print("\n✅ Model ready for inference. Jump to Cell 16 for visualisation + audio.")

## 15 · Main dynamic 250-epoch training loop

In [ ]:
# ============================================================
# BLOCK 15 — Main training loop (auto-resumes from start_epoch)
# ============================================================

best_score = float("inf")

for epoch in range(start_epoch, CFG["total_epochs"] + 1):
    phase = get_phase(epoch)
    set_lr(opt_G, CFG["lr_G"], phase["lr_G_scale"])
    set_lr(opt_D, CFG["lr_D"], phase["lr_D_scale"])

    if epoch == CFG["phase1_epochs"] + 1 and content_teacher is None:
        content_teacher = make_content_teacher()
        print("Created frozen content teacher after Phase 1.")

    G.train(); D.train()
    totals = defaultdict(float); seen = 0
    tic = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{CFG['total_epochs']} | {phase['name']}")
    for batch in pbar:
        src_mel = batch["src_mel"].to(DEVICE, non_blocking=True)
        src_aux = batch["src_aux"].to(DEVICE, non_blocking=True)
        tgt_mel = batch["tgt_mel"].to(DEVICE, non_blocking=True)
        mask    = batch["mask"].to(DEVICE, non_blocking=True)
        src_emo = batch["src_emo"].to(DEVICE, non_blocking=True)
        tgt_emo = batch["tgt_emo"].to(DEVICE, non_blocking=True)
        spk_id  = batch["spk_id"].to(DEVICE, non_blocking=True)

        desired_mel, desired_emo = (src_mel, src_emo) if phase["mode"] == "reconstruct" else (tgt_mel, tgt_emo)

        # 1) Discriminator
        loss_D = torch.tensor(0.0, device=DEVICE)
        if phase["lambda_adv"] > 0:
            with torch.no_grad():
                fake_det = apply_output_gating(G(src_mel, src_aux, spk_id, desired_emo), mask)
            real_in = add_instance_noise(desired_mel, phase["instance_noise"])
            fake_in = add_instance_noise(fake_det,    phase["instance_noise"])
            pr = D(real_in, desired_emo); pf = D(fake_in, desired_emo)
            loss_D = 0.5 * (F.mse_loss(pr, torch.ones_like(pr)) + F.mse_loss(pf, torch.zeros_like(pf)))
            opt_D.zero_grad(set_to_none=True)
            loss_D.backward()
            nn.utils.clip_grad_norm_(D.parameters(), CFG["grad_clip"])
            opt_D.step()

        # 1b) Online SER — keeps adapting to real targets AND current fakes (v2 fix)
        loss_ser_online = torch.tensor(0.0, device=DEVICE)
        if CFG.get("use_online_ser", False) and phase["mode"] == "convert":
            with torch.no_grad():
                fake_for_ser = apply_output_gating(G(src_mel, src_aux, spk_id, desired_emo), mask)
            SER_ONLINE.train()
            logits_real = SER_ONLINE(desired_mel)
            logits_fake = SER_ONLINE(fake_for_ser)
            loss_ser_online = (F.cross_entropy(logits_real, desired_emo) +
                               F.cross_entropy(logits_fake, desired_emo))
            opt_ser_online.zero_grad(set_to_none=True)
            loss_ser_online.backward()
            nn.utils.clip_grad_norm_(SER_ONLINE.parameters(), CFG["grad_clip"])
            opt_ser_online.step()

        # 2) Generator
        use_grl = CFG.get("use_grl", False) and phase.get("lambda_grl", 0.0) > 0
        if use_grl:
            fake, content_feat = G(src_mel, src_aux, spk_id, desired_emo, return_content=True)
        else:
            fake = G(src_mel, src_aux, spk_id, desired_emo)
        fake = apply_output_gating(fake, mask)

        loss_l1     = masked_l1_loss(fake, desired_mel, mask)
        loss_energy = masked_l1_loss_1d(mel_frame_energy(fake), mel_frame_energy(desired_mel), mask)

        loss_content = torch.tensor(0.0, device=DEVICE)
        if content_teacher is not None and phase["lambda_content"] > 0:
            fake_c = content_teacher(fake)
            with torch.no_grad(): src_c = content_teacher(src_mel)
            loss_content = masked_l1_loss(fake_c, src_c, mask)

        loss_cycle = torch.tensor(0.0, device=DEVICE)
        if phase["lambda_cycle"] > 0:
            cyc = apply_output_gating(G(fake, src_aux, spk_id, src_emo), mask)
            loss_cycle = masked_l1_loss(cyc, src_mel, mask)

        # SER loss — frozen reference + honest online SER (v2 fix)
        loss_ser = torch.tensor(0.0, device=DEVICE)
        if phase["lambda_ser"] > 0:
            loss_ser = F.cross_entropy(SER(fake), desired_emo)
            if CFG.get("use_online_ser", False):
                SER_ONLINE.eval()
                loss_ser = 0.5 * loss_ser + 0.5 * F.cross_entropy(SER_ONLINE(fake), desired_emo)

        # Prosody-statistics loss (v2 fix)
        loss_prosody = torch.tensor(0.0, device=DEVICE)
        if phase.get("lambda_prosody", 0.0) > 0:
            loss_prosody = prosody_loss(fake, batch["tgt_aux"].to(DEVICE), mask)

        # GRL disentanglement on content (v2 fix)
        loss_grl = torch.tensor(0.0, device=DEVICE)
        if use_grl:
            emo_logits_from_content = EMO_FROM_CONTENT(content_feat, alpha=CFG.get("grl_alpha", 1.0))
            loss_grl = F.cross_entropy(emo_logits_from_content, src_emo)

        loss_advG = torch.tensor(0.0, device=DEVICE)
        if phase["lambda_adv"] > 0:
            pfG = D(fake, desired_emo); loss_advG = F.mse_loss(pfG, torch.ones_like(pfG))

        loss_G = (phase["lambda_l1"]*loss_l1 + phase["lambda_energy"]*loss_energy +
                  phase["lambda_content"]*loss_content + phase["lambda_cycle"]*loss_cycle +
                  phase["lambda_ser"]*loss_ser + phase["lambda_adv"]*loss_advG +
                  phase.get("lambda_prosody", 0.0)*loss_prosody +
                  phase.get("lambda_grl", 0.0)*loss_grl)

        opt_G.zero_grad(set_to_none=True)
        if use_grl:
            opt_grl.zero_grad(set_to_none=True)
        loss_G.backward()
        nn.utils.clip_grad_norm_(G.parameters(), CFG["grad_clip"])
        opt_G.step()
        if use_grl:
            nn.utils.clip_grad_norm_(EMO_FROM_CONTENT.parameters(), CFG["grad_clip"])
            opt_grl.step()

        bs = src_mel.size(0); seen += bs
        for k, v in {"train_total": loss_G, "train_l1": loss_l1, "train_energy": loss_energy,
                     "train_content": loss_content, "train_cycle": loss_cycle,
                     "train_ser": loss_ser, "train_advG": loss_advG, "train_D": loss_D,
                     "train_prosody": loss_prosody, "train_grl": loss_grl,
                     "train_ser_online": loss_ser_online,
                     "valid_frame_ratio": mask.mean()}.items():
            totals[k] += float(v.item()) * bs

        pbar.set_postfix({
            "G": f"{loss_G.item():.2f}", "l1": f"{loss_l1.item():.2f}",
            "ser": f"{loss_ser.item():.2f}", "pros": f"{loss_prosody.item():.2f}",
            "grl": f"{loss_grl.item():.2f}", "adv": f"{loss_advG.item():.2f}",
            "oSER": f"{loss_ser_online.item():.2f}",
        })

    train_metrics = {k: v / max(1, seen) for k, v in totals.items()}
    val_metrics   = evaluate_epoch(epoch, max_batches=None)

    row = {"epoch": epoch, "phase": phase["name"], "time_sec": time.time() - tic,
           **{k: round(v, 6) for k, v in train_metrics.items()},
           **{k: round(v, 6) for k, v in val_metrics.items()},
           **{f"w_{k}": v for k, v in phase.items() if k.startswith("lambda_")}}
    history.append(row)
    pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)

    score = (val_metrics.get("val_total", 999) +
             0.5 * val_metrics.get("val_content", 0) -
             0.2 * val_metrics.get("val_ser_acc", 0))
    if score < best_score:
        best_score = score
        save_checkpoint(epoch, tag="best_balance", extra={"best_score": best_score})
        print(f"  ✓ saved best_balance (score={best_score:.4f})")

    if epoch % CFG["save_every"] == 0 or epoch in [CFG["phase1_epochs"], CFG["total_epochs"]]:
        path = save_checkpoint(epoch, tag=f"epoch_{epoch:03d}")
        print(f"  ✓ saved {path}")

    save_checkpoint(epoch, tag="latest")

    print(f"Epoch {epoch:03d} | train_total={train_metrics.get('train_total',0):.4f} | "
          f"val_total={val_metrics.get('val_total',0):.4f} | val_l1={val_metrics.get('val_l1',0):.4f} | "
          f"val_ser_acc={val_metrics.get('val_ser_acc',0):.3f} | time={row['time_sec']:.1f}s")

print("\n🎉 Training complete.")

## 16 · Plot training curves

In [ ]:
# ============================================================
# BLOCK 16 — Plot training curves
# ============================================================
plot_history()

## 17 · Inference, vocoder fallback, audio playback + visual comparison

In [ ]:
# ============================================================
# BLOCK 17 — Inference helpers + visual/audio comparison
# ============================================================

def mel_db_to_audio_griffinlim(mel_db):
    mel_power = librosa.db_to_power(mel_db, ref=1.0)
    wav = librosa.feature.inverse.mel_to_audio(
        M=mel_power, sr=CFG["sample_rate"], n_fft=CFG["n_fft"],
        hop_length=CFG["hop_length"], win_length=CFG["win_length"],
        fmin=CFG["fmin"], fmax=CFG["fmax"], power=2.0, n_iter=64)
    wav = wav.astype(np.float32)
    if np.max(np.abs(wav)) > 0:
        wav = wav / (np.max(np.abs(wav)) + 1e-8) * 0.95
    return wav

def load_real_wav_from_row(row):
    if COL_WAV is None: return None
    p = resolve_path(row[COL_WAV])
    if p is None or not Path(p).exists(): return None
    try:
        wav, _ = librosa.load(p, sr=CFG["sample_rate"], mono=True)
        return wav.astype(np.float32)
    except Exception:
        return None

def estimate_f0_from_wav(wav):
    try:
        f0, _, _ = librosa.pyin(wav, fmin=50, fmax=500, sr=CFG["sample_rate"],
                                 frame_length=CFG["win_length"], hop_length=CFG["hop_length"])
        return np.nan_to_num(f0, nan=0.0).astype(np.float32)
    except Exception:
        return np.zeros(1, dtype=np.float32)

@torch.no_grad()
def generate_from_pair(ds, idx):
    G.eval()
    b = ds[idx]
    src_mel = b["src_mel"].unsqueeze(0).to(DEVICE)
    src_aux = b["src_aux"].unsqueeze(0).to(DEVICE)
    spk_id  = b["spk_id"].view(1).to(DEVICE)
    tgt_emo = b["tgt_emo"].view(1).to(DEVICE)
    fake_norm = G(src_mel, src_aux, spk_id, tgt_emo)[0].detach().cpu().numpy()
    valid_T = b["src_mel"].shape[-1]
    src_db = denormalize_mel(b["src_mel"].numpy())[:, :valid_T]
    tgt_db = denormalize_mel(b["tgt_mel"].numpy())[:, :valid_T]
    gen_db = denormalize_mel(fake_norm)[:, :valid_T]
    src_aux_np = b["src_aux"].numpy()[:, :valid_T]
    tgt_aux_np = b["tgt_aux"].numpy()[:, :valid_T]
    src_f0 = denormalize_f0(src_aux_np[0], src_aux_np[2], STATS)
    tgt_f0 = denormalize_f0(tgt_aux_np[0], tgt_aux_np[2], STATS)
    src_e  = denormalize_energy(src_aux_np[1], STATS)
    tgt_e  = denormalize_energy(tgt_aux_np[1], STATS)
    gen_e  = derive_energy_from_mel_db(gen_db)
    pair_row = ds.pairs_df.iloc[int(b["pair_index"])]
    src_row  = df_work.iloc[int(pair_row["src_idx"])]
    tgt_row  = df_work.iloc[int(pair_row["tgt_idx"])]
    src_wav_real = load_real_wav_from_row(src_row)
    tgt_wav_real = load_real_wav_from_row(tgt_row)
    src_wav = src_wav_real if src_wav_real is not None else mel_db_to_audio_griffinlim(src_db)
    tgt_wav = tgt_wav_real if tgt_wav_real is not None else mel_db_to_audio_griffinlim(tgt_db)
    gen_wav = mel_db_to_audio_griffinlim(gen_db)
    gen_f0  = estimate_f0_from_wav(gen_wav)
    return {"src_db": src_db, "tgt_db": tgt_db, "gen_db": gen_db,
            "src_f0": src_f0, "tgt_f0": tgt_f0, "gen_f0": gen_f0,
            "src_energy": src_e, "tgt_energy": tgt_e, "gen_energy": gen_e,
            "src_wav": src_wav, "tgt_wav": tgt_wav, "gen_wav": gen_wav,
            "src_emotion": id_to_emotion[int(b["src_emo"])],
            "tgt_emotion": id_to_emotion[int(b["tgt_emo"])],
            "speaker": id_to_speaker[int(b["spk_id"])]}

def compare_generated(idx=0, split="val", save_prefix=None):
    ds = val_ds if split == "val" else train_ds
    out = generate_from_pair(ds, idx)
    title = f"Speaker={out['speaker']} | {out['src_emotion']} → {out['tgt_emotion']}"
    fig, axes = plt.subplots(1, 3, figsize=(20, 4))
    plot_mel(axes[0], out["src_db"], "Source neutral mel")
    plot_mel(axes[1], out["tgt_db"], "Ground-truth target mel")
    plot_mel(axes[2], out["gen_db"], "Generated target-emotion mel")
    fig.suptitle(title); plt.tight_layout()
    if save_prefix: fig.savefig(PLOT_DIR / f"{save_prefix}_mel.png", dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(15, 4))
    t_src = np.arange(len(out["src_f0"])) * CFG["hop_length"] / CFG["sample_rate"]
    t_tgt = np.arange(len(out["tgt_f0"])) * CFG["hop_length"] / CFG["sample_rate"]
    t_gen = np.arange(len(out["gen_f0"])) * CFG["hop_length"] / CFG["sample_rate"]
    plt.plot(t_src, out["src_f0"], label="source F0")
    plt.plot(t_tgt, out["tgt_f0"], label="ground-truth target F0")
    plt.plot(t_gen, out["gen_f0"], label="generated F0")
    plt.title("F0 comparison"); plt.xlabel("Time (s)"); plt.ylabel("Hz")
    plt.grid(True, alpha=0.3); plt.legend()
    if save_prefix: plt.savefig(PLOT_DIR / f"{save_prefix}_f0.png", dpi=150, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(15, 4))
    plt.plot(np.arange(len(out["src_energy"])) * CFG["hop_length"] / CFG["sample_rate"], out["src_energy"], label="source")
    plt.plot(np.arange(len(out["tgt_energy"])) * CFG["hop_length"] / CFG["sample_rate"], out["tgt_energy"], label="ground truth")
    plt.plot(np.arange(len(out["gen_energy"])) * CFG["hop_length"] / CFG["sample_rate"], out["gen_energy"], label="generated")
    plt.title("Energy comparison"); plt.xlabel("Time (s)"); plt.ylabel("dB-like energy")
    plt.grid(True, alpha=0.3); plt.legend()
    if save_prefix: plt.savefig(PLOT_DIR / f"{save_prefix}_energy.png", dpi=150, bbox_inches="tight")
    plt.show()

    if save_prefix is None: save_prefix = f"{split}_{idx:03d}"
    src_path = AUDIO_DIR / f"{save_prefix}_source_{out['src_emotion']}.wav"
    tgt_path = AUDIO_DIR / f"{save_prefix}_groundtruth_{out['tgt_emotion']}.wav"
    gen_path = AUDIO_DIR / f"{save_prefix}_generated_{out['tgt_emotion']}.wav"
    sf.write(src_path, out["src_wav"], CFG["sample_rate"])
    sf.write(tgt_path, out["tgt_wav"], CFG["sample_rate"])
    sf.write(gen_path, out["gen_wav"], CFG["sample_rate"])

    print(f"\n🔊 SOURCE AUDIO ({out['src_emotion']}):")
    display(Audio(str(src_path), rate=CFG["sample_rate"]))
    print(f"🔊 GROUND-TRUTH TARGET AUDIO ({out['tgt_emotion']}):")
    display(Audio(str(tgt_path), rate=CFG["sample_rate"]))
    print(f"🔊 GENERATED AUDIO ({out['tgt_emotion']}):")
    display(Audio(str(gen_path), rate=CFG["sample_rate"]))

    with torch.no_grad():
        gm = torch.tensor(normalize_mel(out["gen_db"])[None], dtype=torch.float32).to(DEVICE)
        prob = torch.softmax(SER(gm), dim=1)[0].detach().cpu().numpy()
        pred = int(prob.argmax())
    print(f"\nSER prediction on generated mel: {id_to_emotion[pred]} ({prob[pred]:.3f})")
    print("Audio files saved to:", AUDIO_DIR)

# Run 3 validation examples
for i in range(min(3, len(val_ds))):
    compare_generated(i, split="val", save_prefix=f"final_val_{i:03d}")

## 18 · Batch evaluation — HONEST emotion metrics (v2)

> ⚠️ **The frozen-SER confusion matrix is NOT a valid success metric.** A generator can
> hit ~100% on it while injecting zero audible emotion. This cell reports what matters:
>
> 1. **Prosody drift** — does generated energy move toward the target emotion (and away
>    from neutral)? Computed from the signal, so it can't be faked.
> 2. **Online-SER accuracy** — a classifier that kept adapting. If the generator was
>    cheating, this is much lower than the frozen-SER number.
> 3. **Frozen-SER accuracy** — kept only for comparison with the old result.
>
> Read the printed **DIAGNOSIS** line.

In [ ]:
# ============================================================
# BLOCK 18 — Honest batch evaluation (prosody + online SER + frozen SER)
# ============================================================

def rmse(a, b):
    T = min(a.shape[-1], b.shape[-1])
    return float(np.sqrt(np.mean((a[..., :T] - b[..., :T]) ** 2)))

def l1_np(a, b):
    T = min(a.shape[-1], b.shape[-1])
    return float(np.mean(np.abs(a[..., :T] - b[..., :T])))

def _stats_np(x):
    x = np.asarray(x, dtype=np.float64)
    if len(x) < 2:
        return 0.0, 0.0, 0.0
    return float(x.mean()), float(x.std()), float(np.mean(np.abs(np.diff(x))))

@torch.no_grad()
def honest_batch_eval(ds, max_items=60):
    rows = []
    G.eval(); SER.eval()
    if CFG.get("use_online_ser", False): SER_ONLINE.eval()
    n = min(max_items, len(ds))

    for idx in tqdm(range(n), desc="Honest evaluation"):
        out = generate_from_pair(ds, idx)

        src_e_mean, src_e_std, src_e_dyn = _stats_np(out["src_energy"])
        tgt_e_mean, tgt_e_std, tgt_e_dyn = _stats_np(out["tgt_energy"])
        gen_e_mean, gen_e_std, gen_e_dyn = _stats_np(out["gen_energy"])

        def f0_stats(f0):
            v = f0[f0 > 0]
            return _stats_np(v) if len(v) > 1 else (0.0, 0.0, 0.0)
        src_f0_mean, _, _ = f0_stats(out["src_f0"])
        tgt_f0_mean, _, _ = f0_stats(out["tgt_f0"])
        gen_f0_mean, _, _ = f0_stats(out["gen_f0"])

        moved_energy     = int(abs(gen_e_mean - tgt_e_mean) < abs(src_e_mean - tgt_e_mean))
        moved_energy_dyn = int(abs(gen_e_dyn - tgt_e_dyn)  < abs(src_e_dyn - tgt_e_dyn))

        gm = torch.tensor(normalize_mel(out["gen_db"])[None], dtype=torch.float32).to(DEVICE)
        target_id = emotion_to_id[out["tgt_emotion"]]
        frozen_pred = int(torch.softmax(SER(gm), dim=1)[0].argmax())
        if CFG.get("use_online_ser", False):
            online_pred = int(torch.softmax(SER_ONLINE(gm), dim=1)[0].argmax())
        else:
            online_pred = frozen_pred

        rows.append({
            "idx": idx, "speaker": out["speaker"], "target_emotion": out["tgt_emotion"],
            "frozen_ser_pred": id_to_emotion[frozen_pred], "frozen_correct": int(frozen_pred == target_id),
            "online_ser_pred": id_to_emotion[online_pred], "online_correct": int(online_pred == target_id),
            "moved_energy_toward_tgt": moved_energy, "moved_energy_dyn_toward_tgt": moved_energy_dyn,
            "src_e_mean": round(src_e_mean,3), "gen_e_mean": round(gen_e_mean,3), "tgt_e_mean": round(tgt_e_mean,3),
            "src_e_dyn": round(src_e_dyn,3), "gen_e_dyn": round(gen_e_dyn,3), "tgt_e_dyn": round(tgt_e_dyn,3),
            "src_f0_mean": round(src_f0_mean,1), "gen_f0_mean": round(gen_f0_mean,1), "tgt_f0_mean": round(tgt_f0_mean,1),
            "mel_l1_vs_tgt": round(l1_np(out["gen_db"], out["tgt_db"]),3),
            "mel_l1_vs_src": round(l1_np(out["gen_db"], out["src_db"]),3),
        })
    return pd.DataFrame(rows)

eval_df = honest_batch_eval(val_ds, max_items=min(60, len(val_ds)))
display(eval_df.head(12))
eval_path = OUT_DIR / "honest_evaluation.csv"
eval_df.to_csv(eval_path, index=False)
print("Saved:", eval_path)

if len(eval_df):
    print("\n" + "="*60)
    print("  HONEST EMOTION-INJECTION REPORT")
    print("="*60)
    frozen_acc = eval_df["frozen_correct"].mean()
    online_acc = eval_df["online_correct"].mean()
    moved      = eval_df["moved_energy_toward_tgt"].mean()
    moved_dyn  = eval_df["moved_energy_dyn_toward_tgt"].mean()
    print(f"  Frozen-SER accuracy   : {frozen_acc:.3f}   (the OLD, misleading metric)")
    print(f"  Online-SER accuracy   : {online_acc:.3f}   (honest classifier)")
    print(f"  Energy moved -> target : {moved:.3f}")
    print(f"  Energy-dyn moved -> tgt: {moved_dyn:.3f}")
    print("-"*60)
    if frozen_acc > 0.85 and online_acc < 0.55:
        print("  ! DIAGNOSIS: Generator likely STILL EXPLOITING the frozen SER.")
        print("    Raise lambda_prosody / lambda_adv, or online_ser_warmup.")
    elif online_acc >= 0.6 and moved >= 0.6:
        print("  OK DIAGNOSIS: Emotion injection looks REAL (honest acc + prosody move).")
    else:
        print("  ~ DIAGNOSIS: Partial. Train longer or nudge lambda_ser / lambda_prosody up.")
    print("="*60)
    print("\nPer-target-emotion honest accuracy:")
    display(eval_df.groupby("target_emotion")[["frozen_correct","online_correct","moved_energy_toward_tgt"]].mean().round(3))

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
if len(eval_df):
    labels_order = [id_to_emotion[i] for i in sorted(id_to_emotion)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, predcol, title in [(axes[0], "frozen_ser_pred", "Frozen SER (misleading)"),
                                (axes[1], "online_ser_pred", "Online SER (honest)")]:
        cm = confusion_matrix(eval_df["target_emotion"], eval_df[predcol], labels=labels_order)
        ConfusionMatrixDisplay(cm, display_labels=labels_order).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(title)
    plt.tight_layout(); plt.savefig(PLOT_DIR / "honest_vs_frozen_confusion.png", dpi=150, bbox_inches="tight"); plt.show()

    fig, ax = plt.subplots(figsize=(10, 5))
    for emo in labels_order:
        sub = eval_df[eval_df["target_emotion"] == emo]
        if len(sub) == 0: continue
        ax.scatter(sub["src_e_mean"], sub["gen_e_mean"], label=f"{emo} (gen)", alpha=0.7, s=40)
    lim = [eval_df[["src_e_mean","gen_e_mean","tgt_e_mean"]].min().min(),
           eval_df[["src_e_mean","gen_e_mean","tgt_e_mean"]].max().max()]
    ax.plot(lim, lim, "k--", alpha=0.4, label="no-change line")
    ax.set_xlabel("Source (neutral) energy mean"); ax.set_ylabel("Generated energy mean")
    ax.set_title("Prosody drift: points OFF the dashed line = energy actually changed")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(PLOT_DIR / "prosody_drift.png", dpi=150, bbox_inches="tight"); plt.show()


In [ ]:
# ============================================================
# QUICK LISTEN — play generated audio for a validation sample
# ============================================================
# Change idx to hear different samples. Set split="train" for training pairs.

def listen(idx=0, split="val"):
    ds = val_ds if split == "val" else train_ds
    out = generate_from_pair(ds, idx)

    print(f"Speaker: {out['speaker']}  |  {out['src_emotion']} → {out['tgt_emotion']}")

    # Save the three audio versions
    src_path = AUDIO_DIR / f"listen_{split}_{idx}_source_{out['src_emotion']}.wav"
    tgt_path = AUDIO_DIR / f"listen_{split}_{idx}_groundtruth_{out['tgt_emotion']}.wav"
    gen_path = AUDIO_DIR / f"listen_{split}_{idx}_generated_{out['tgt_emotion']}.wav"
    sf.write(src_path, out["src_wav"], CFG["sample_rate"])
    sf.write(tgt_path, out["tgt_wav"], CFG["sample_rate"])
    sf.write(gen_path, out["gen_wav"], CFG["sample_rate"])

    print(f"\n🔊 SOURCE ({out['src_emotion']}):")
    display(Audio(str(src_path), rate=CFG["sample_rate"]))
    print(f"🔊 GROUND-TRUTH TARGET ({out['tgt_emotion']}):")
    display(Audio(str(tgt_path), rate=CFG["sample_rate"]))
    print(f"🔊 GENERATED ({out['tgt_emotion']}):")
    display(Audio(str(gen_path), rate=CFG["sample_rate"]))

    # SER prediction on the generated mel
    with torch.no_grad():
        gm = torch.tensor(normalize_mel(out["gen_db"])[None], dtype=torch.float32).to(DEVICE)
        prob = torch.softmax(SER(gm), dim=1)[0].cpu().numpy()
        pred = int(prob.argmax())
    print(f"\nSER on generated: {id_to_emotion[pred]} ({prob[pred]:.3f})")

# Play a few
for i in range(3):
    listen(i, split="val")
    print("─" * 50)

## 19 · Export results as ZIP for download

In [ ]:
# ============================================================
# BLOCK 19 — Export results
# ============================================================

zip_base = WORK_ROOT / "evc_clean_250_results"
zip_path = zip_base.with_suffix(".zip")
if zip_path.exists(): zip_path.unlink()

shutil.make_archive(str(zip_base), "zip", OUT_DIR)
mb = zip_path.stat().st_size / 1e6
print(f"✅ Created: {zip_path}  ({mb:.1f} MB)")
print("Download from the Kaggle Output tab on the right.")